# Load packages

In [1]:
# -------------------------
# Standard library
# -------------------------
import os
import glob
import time
import re
import warnings
import logging
import gc
from datetime import datetime, timedelta
from pathlib import Path
import shutil

# -------------------------
# Numerical / scientific
# -------------------------
import numpy as np
import pandas as pd
import xarray as xr
import gsw
from netCDF4 import Dataset

from scipy.stats import linregress, median_abs_deviation, norm
from scipy.interpolate import PchipInterpolator
from scipy.optimize import least_squares, curve_fit

from numpy.lib.stride_tricks import sliding_window_view

# -------------------------
# Machine learning / stats
# -------------------------
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score, mean_absolute_error
from sklearn.linear_model import LinearRegression, HuberRegressor
from sklearn.exceptions import InconsistentVersionWarning

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

# -------------------------
# Plotting
# -------------------------
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import ticker, dates, patches
from matplotlib.colors import LogNorm
import seaborn as sns
import cmocean as cmo

import cartopy
import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER

# -------------------------
# Utilities
# -------------------------
from tqdm.std import tqdm
import timezonefinder
import pytz
tf = timezonefinder.TimezoneFinder()

import pvlib
import import_ipynb
import copernicusmarine
logging.getLogger().setLevel(logging.ERROR)
logging.getLogger("copernicusmarine").setLevel(logging.ERROR)
logging.getLogger("copernicus_marine_client").setLevel(logging.ERROR)

# -------------------------
# Functions designed for this project
# -------------------------
%load_ext autoreload
%autoreload 2

import bgc_argo_utils as bgc

# Set matplotlib parameters
mpl.rcParams['figure.facecolor'] = '1'
mpl.rcParams['mathtext.default'] = 'regular'

# Set font parameters
font = {'family' : 'sans-serif',
        'weight' : 'normal',
        'size'   : 10}
mpl.rc('font', **font)

/opt/miniconda3/envs/pyth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Data Processing

## Define directories

In [2]:
root = '/Volumes/Tommy/BGC-ARGO'

# This loads in a biomes file from Fay & McKinley 2014 - if you wish to look at regional variability
biomes = xr.open_dataset('../DATA/Time_Varying_Biomes_25KM.nc').MeanBiomes

## Load in SOCA Machine Learning Model

In [3]:
from FUNCTIONS.OTHER_FUNCTIONS import rad_date, rad_LC_hour, derive_ZPD
from FUNCTIONS.PAR_ADJUSTED_FUNCTION import PAR_ADJUSTED_profiles
from FUNCTIONS.ED380_ADJUSTED_FUNCTION import ED380_ADJUSTED_profiles
from FUNCTIONS.ED412_ADJUSTED_FUNCTION import ED412_ADJUSTED_profiles
from FUNCTIONS.ED490_ADJUSTED_FUNCTION import ED490_ADJUSTED_profiles

## Access Remote Sensing Data

### PAR

In [4]:
# daily PAR data is saved offline to speed things
fname = '/Volumes/Tommy/GLOBCOLOUR/PAR_compressed/*.nc'
par_list = sorted(glob.glob(fname))

# Precompute once
par_dates = np.array([bgc.extract_date(fi) for fi in par_list])
par_sort_idx = np.argsort(par_dates)

sorted_par_files = np.array(par_list)[par_sort_idx]
sorted_par_dates = par_dates[par_sort_idx]

### GlobColour

In [6]:
# user must put their copernicus username and password here once to store them
copernicusmarine.login(username="", password="")

True

In [5]:
meta = copernicusmarine.open_dataset(
    "cmems_obs-oc_glo_bgc-reflectance_my_l3-multi-4km_P1D"
)
times = meta["time"].values
DATA_START = pd.to_datetime(times[0])
DATA_END = pd.to_datetime(times[-1])
meta.close()

## Construct file flists to process BGC-Argo floats

This is run upon first time using the notebook.

In [5]:
new_root_name = "202605-BgcArgoSprof"

base_dir = Path(root) / new_root_name
fname = str(base_dir / "dac" / "*" / "*.nc")

reject_dir = base_dir / "_rejected_missing_required"
reject_dir.mkdir(exist_ok=True)

flist = sorted(
    f for f in glob.glob(fname)
    if not Path(f).name.startswith("._")
)

flist_with_par = []
flist_without_par = []
failed_files = []
skipped_missing_required = []

for file in tqdm(flist, desc="Checking Sprof files", dynamic_ncols=True):

    file = Path(file)

    try:
        with xr.open_dataset(file, decode_times=False) as xds_new:

            data_vars = set(xds_new.data_vars)

            has_required = (
                "BBP700" in data_vars
                and (
                    "CHLA" in data_vars
                    or "CHLA_ADJUSTED" in data_vars
                    or "CHLA_FLUORESCENCE" in data_vars
                    or "CHLA_FLUORESCENCE_ADJUSTED" in data_vars
                )
            )

            has_par = (
                "DOWNWELLING_PAR" in data_vars
                and "DOWN_IRRADIANCE490" in data_vars
            )

        if not has_required:
            skipped_missing_required.append(str(file))

            # Preserve DAC subdirectory structure inside reject folder
            rel = file.relative_to(base_dir)
            target = reject_dir / rel
            target.parent.mkdir(parents=True, exist_ok=True)

            shutil.move(str(file), str(target))
            continue

        if has_par:
            flist_with_par.append(str(file))
        else:
            flist_without_par.append(str(file))

    except Exception as e:
        failed_files.append((str(file), str(e)))
        print(f"Failed checking {file.name}: {e}")


flist_with_par = sorted(flist_with_par)
flist_without_par = sorted(flist_without_par)

print(f"Total candidate files       = {len(flist)}")
print(f"Files with PAR/irradiance   = {len(flist_with_par)}")
print(f"Files without irradiance    = {len(flist_without_par)}")
print(f"Moved missing required vars = {len(skipped_missing_required)}")
print(f"Failed files                = {len(failed_files)}")
print(f"Rejected files moved to     = {reject_dir}")

Checking Sprof files: 100%|█████████████████████████████████████████████████████████| 1422/1422 [01:42<00:00, 13.85it/s]

Total candidate files       = 1422
Files with PAR/irradiance   = 386
Files without irradiance    = 1036
Moved missing required vars = 0
Failed files                = 0
Rejected files moved to     = /Volumes/Tommy/BGC-ARGO/202605-BgcArgoSprof/_rejected_missing_required


## Processing Floats

### Measured - Floats with Irradiance Sensors

In [6]:
skipped_files = []

outdir = Path(f"{resdir}/Output Final/Profiles_SOCA/ROESLER/MEASURED")
outdir_npp = Path(f"{resdir}/Output Final/NPP_SOCA/ROESLER/MEASURED")

# create directories (including parents) if they don't exist
outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist_with_par):
    
    # Get the output file path
    out_file = os.path.join(outdir, os.path.basename(file))  # You can modify this if you want a different naming convention
    
    # Check if the file has already been processed
    if os.path.exists(out_file):
        #print(f"Skipping {file}, already processed.")
        continue  # Skip this file and move to the next one

    name = os.path.basename(file).split('.')[0]
    base_vars = [
            'PRES', 'PRES_QC',
            'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC',
            'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC',
            'BBP700', 'BBP700_QC', 'BBP700_ADJUSTED', 'BBP700_ADJUSTED_QC',
            'DOWNWELLING_PAR', 'DOWNWELLING_PAR_QC']
    
    xds = bgc.process_bgc_argo_file(file, base_vars=base_vars)

    if xds is None:
        print(f"⚠️ Skipping {file} — dataset is None.")
        continue
    
    xds = bgc.clean_range_filter(xds, 'BBP700', 0, 0.05)
    # (Applies Dall’Olmo 2012 spectral slope of -0.78)
    xds['BBP470'] = xds['BBP700'] * (470 / 700) ** -0.78
    
    # Compute non-algal (NAP) and phytoplankton (PHY) fractions using Stoer & Fennel
    xds = bgc.calculate_bbp_nap_and_phyto(xds, bbp_var='BBP470', depth_var='PRES')

    if xds['BBP_PHY'].isnull().all():
        print(f"{name}: no valid BBP_PHY after QC/filtering – skipping.")
        skipped_files.append(name)
        continue

    # Convert bbp_phy to Cphyto using slope factor from Graff et al. (2015)
    xds['CPHYTO'] = 12128 * xds['BBP_PHY']
    xds["CPHYTO"] = xds["CPHYTO"].where(np.isfinite(xds["CPHYTO"]) & (xds["CPHYTO"] >= 0))
        
    # Ensure metadata are present
    required_meta = ['JULD', 'LATITUDE', 'LONGITUDE']
    for m in required_meta:
        if m not in xds:
            print(f"{name}: missing metadata field {m}; skipping.")
            continue
    
    xds = xds.dropna(dim='N_PROF', subset=required_meta)
    xds = xds.dropna(dim='N_PROF', how='all', subset=xds.data_vars)
    
    # Require at least 2 profiles for temporal/vertical context
    if xds.sizes.get('N_PROF', 0) <= 1: 
        name = os.path.basename(file)
        print(f"{name}: fails final QC – not enough valid profiles.")
        continue
    
    else:
        xds['PSAL_ABSOLUTE'] = gsw.SA_from_SP(xds.PSAL, xds.PRES, xds.LONGITUDE, xds.LATITUDE)
        xds['DENSITY'] = gsw.pot_rho_t_exact(xds.PSAL_ABSOLUTE, xds.TEMP, xds.PRES, 0)
        xds["MLD"] = bgc.compute_mld_density_threshold(xds, rho_name="DENSITY", depth_name="PRES", ref_depth=10, delta_rho=0.03)

        # Euphotic depth calculations
        par = xds['DOWNWELLING_PAR'].sel(PRES=slice(0, 199))
        depth = xds['PRES'].sel(PRES=slice(0, 199))
    
        slope, r2, n, zeu = xr.apply_ufunc(
            bgc.euphotic_depth,
            par,
            depth,
            input_core_dims=[['PRES'], ['PRES']],  
            output_core_dims=[[], [], [], []],      
            vectorize=True,
            dask='parallelized',
            output_dtypes=[float, float, float, float],
            kwargs={'initial_fit_depth': 100, 'max_fit_depth': 186, 'step': 20})
        
        xds = xds.assign(
            KD_PAR=slope,
            KD_PAR_R2=r2,
            ZEU=zeu)
    
        # Chlorophyll fluorescence quenching
        xds["SOLAR_ELEVATION"] = bgc.calculate_solar_elevation(xds)

        ds = xds.sel(PRES=slice(0, 199))
        result = xr.apply_ufunc(bgc.apply_xing_quenching_correction,
                                ds['CHLA_FLUORESCENCE'],
                                ds['BBP_PHY'],
                                ds['DOWNWELLING_PAR'],
                                ds['MLD'],
                                ds['SOLAR_ELEVATION'],
                                input_core_dims=[['PRES'], ['PRES'], ['PRES'], [], []],
                                output_core_dims=[['PRES'], ['PRES'], [], []],
                                vectorize=True,
                                dask='parallelized',
                                output_dtypes=[float, float, float, bool])
    
        xds = xds.assign(CHLA= result[0],
                         NPQ=result[1],
                         QUENCHING_DEPTH=result[2],
                         QUENCHING_FLAG=result[3])
    
        xds['CHLA'] = xds.CHLA / 2
        xds = bgc.clean_range_filter(xds, 'CHLA', 0.014, 50)

        # Calculate daily PAR from instantaneous PAR using a clear sky model
        xds["PAR_DAILY"] = bgc.compute_daily_par_clear_sky(xds, dt_seconds=60, par_top_depth=10)
        
        # Apply NPP algorithms
        ds = xds.sel(PRES=slice(0, 199))
        result = xr.apply_ufunc(bgc.cbpm_argo,
                                ds['CHLA'],
                                ds['CPHYTO'],
                                ds['PAR_DAILY'],
                                ds['DATETIME'].dt.dayofyear,
                                ds['LATITUDE'],
                                input_core_dims=[['PRES'], ['PRES'], [], [], []],
                                output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                                vectorize=True,
                                dask='parallelized',
                                output_dtypes=[float] * 4)
    
        xds = xds.assign(WESTBERRY_CBPM=result[0],
                         WESTBERRY_MU=result[1],
                         WESTBERRY_NUT_TEMP_FUNC=result[2],
                         WESTBERRY_IG_FUNC=result[3])
    
        xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] > 0, np.nan)
        
        var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM', 'WESTBERRY_NUT_TEMP_FUNC', 'WESTBERRY_IG_FUNC']
        
        # Integrate to 199 m
        int_all = bgc.integrate_along_pres_multiple_vars(
            xds.where(xds.PRES <= 199, drop=False), var_list
        ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Integrate to Zeu
        zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
        int_zeu = bgc.integrate_along_pres_multiple_vars(
            xds.where(zeu_mask, drop=False), var_list
        ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Integrate to MLD
        mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
        int_mld = bgc.integrate_along_pres_multiple_vars(
            xds.where(mld_mask, drop=False), var_list
        ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Merge safely
        int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')
    
        data_eu = xds[['WESTBERRY_CBPM', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
        int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
        int_all['ZEU_CHLA'] = data_eu['CHLA']
        int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
        int_all['ZEU_TEMP'] = data_eu['TEMP']
        int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']
    
        data_mld = xds[['WESTBERRY_CBPM', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
        int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
        int_all['MLD_CHLA'] = data_mld['CHLA']
        int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
        int_all['MLD_TEMP'] = data_mld['TEMP']
        int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']
    
        od = abs(1 / xds.KD_PAR)
        data_od = xds[['WESTBERRY_CBPM', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
        int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
        int_all['OD1_CHLA'] = data_od['CHLA']
        int_all['OD1_CPHYTO'] = data_od['CPHYTO']
        int_all['OD1_TEMP'] = data_od['TEMP']
        int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
        
        int_all['DATETIME'] = xds.DATETIME
        int_all['LATITUDE'] = xds.LATITUDE
        int_all['LONGITUDE'] = xds.LONGITUDE
        int_all['ZEU'] = xds.ZEU
        int_all['MLD'] = xds.MLD
        int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
        int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG
    
        if int_all.N_PROF.shape[0] == 0:
            continue
    
        xds['BIOME'] = biomes.interp(lat=xds['LATITUDE'], lon=xds['LONGITUDE'], method='nearest')
        
        int_all['BIOME'] = biomes.interp(lat=int_all['LATITUDE'], lon=int_all['LONGITUDE'], method='nearest')
    
        xds = xds.drop_vars(['lat', 'lon'])
    
        int_all = int_all.drop_vars(['lat', 'lon'])
    
        xds.to_netcdf(
            f"{outdir}/{name}.nc",
            format="NETCDF4",
            encoding=bgc.make_encoding(xds, complevel=1),
        )
        
        int_all.to_netcdf(
            f"{outdir_npp}/{name}_NPP.nc",
            format="NETCDF4",
            encoding=bgc.make_encoding(int_all, complevel=1),
        )


  9%|███████▋                                                                          | 36/386 [00:04<00:43,  8.03it/s]

3901584_Sprof: no valid BBP_PHY after QC/filtering – skipping.


 13%|██████████▊                                                                       | 51/386 [00:08<00:59,  5.63it/s]

6904185_Sprof: no valid BBP_PHY after QC/filtering – skipping.


 17%|█████████████▌                                                                    | 64/386 [00:20<02:09,  2.48it/s]

...

 85%|█████████████████████████████████████████████████████████████████████            | 329/386 [39:29<07:03,  7.42s/it]

7901007_Sprof: no valid BBP_PHY after QC/filtering – skipping.


 91%|█████████████████████████████████████████████████████████████████████████▋       | 351/386 [41:19<02:10,  3.72s/it]

3902684_Sprof.nc: fails final QC – not enough profiles.
⚠️ Skipping /Volumes/Tommy/BGC-ARGO/202605-BgcArgoSprof/dac/csio/3902684_Sprof.nc — dataset is None.


100%|█████████████████████████████████████████████████████████████████████████████████| 386/386 [44:07<00:00,  6.86s/it]


#### Including Downwelling Irradiance 490

In [7]:
outdir = Path(f"{resdir}/Output Final/Profiles_SOCA/KD490/MEASURED")
outdir_npp = Path(f"{resdir}/Output Final/NPP_SOCA/KD490/MEASURED")

# create directories (including parents) if they don't exist
outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist_with_par):
    
    # Get the output file path
    out_file = os.path.join(outdir, os.path.basename(file))  # You can modify this if you want a different naming convention
    
    # Check if the file has already been processed
    if os.path.exists(out_file):
        #print(f"Skipping {file}, already processed.")
        continue  # Skip this file and move to the next one

    name = os.path.basename(file).split('.')[0]
    base_vars = [
        'PRES', 'PRES_QC',
        'TEMP', 'TEMP_QC', 'TEMP_ADJUSTED', 'TEMP_ADJUSTED_QC',
        'PSAL', 'PSAL_QC', 'PSAL_ADJUSTED', 'PSAL_ADJUSTED_QC',
        'BBP700', 'BBP700_QC', 'BBP700_ADJUSTED', 'BBP700_ADJUSTED_QC',
        'DOWNWELLING_PAR', 'DOWNWELLING_PAR_QC', 
        'DOWN_IRRADIANCE490', 'DOWN_IRRADIANCE490_QC']
    
    xds = bgc.process_bgc_argo_file(file, base_vars=base_vars)

    if xds is None:
        print(f"⚠️ Skipping {file} — dataset is None.")
        continue
    
    xds = bgc.clean_range_filter(xds, 'BBP700', 0, 0.05)
    # (Applies Dall’Olmo 2012 spectral slope of -0.78)
    xds['BBP470'] = xds['BBP700'] * (470 / 700) ** -0.78
    
    # Compute non-algal (NAP) and phytoplankton (PHY) fractions using Stoer & Fennel
    xds = bgc.calculate_bbp_nap_and_phyto(xds, bbp_var='BBP470', depth_var='PRES')

    if xds['BBP_PHY'].isnull().all():
        print(f"{name}: no valid BBP_PHY after QC/filtering – skipping.")
        skipped_files.append(name)
        continue

    # Convert bbp_phy to Cphyto using slope factor from Graff et al. (2015)
    xds['CPHYTO'] = 12128 * xds['BBP_PHY']
    xds["CPHYTO"] = xds["CPHYTO"].where(np.isfinite(xds["CPHYTO"]) & (xds["CPHYTO"] >= 0))
        
    # Ensure metadata are present
    required_meta = ['JULD', 'LATITUDE', 'LONGITUDE']
    for m in required_meta:
        if m not in xds:
            print(f"{name}: missing metadata field {m}; skipping.")
            continue
    
    xds = xds.dropna(dim='N_PROF', subset=required_meta)
    xds = xds.dropna(dim='N_PROF', how='all', subset=xds.data_vars)
    
    # Require at least 2 profiles for temporal/vertical context
    if xds.sizes.get('N_PROF', 0) <= 1: 
        name = os.path.basename(file)
        print(f"{name}: fails final QC – not enough valid profiles.")
        continue
    
    else:
        xds['PSAL_ABSOLUTE'] = gsw.SA_from_SP(xds.PSAL, xds.PRES, xds.LONGITUDE, xds.LATITUDE)
        xds['DENSITY'] = gsw.pot_rho_t_exact(xds.PSAL_ABSOLUTE, xds.TEMP, xds.PRES, 0)
        xds["MLD"] = bgc.compute_mld_density_threshold(xds, rho_name="DENSITY", depth_name="PRES", ref_depth=10, delta_rho=0.03)

        # Euphotic depth calculations
        par = xds['DOWNWELLING_PAR'].sel(PRES=slice(0, 199))
        depth = xds['PRES'].sel(PRES=slice(0, 199))
    
        slope, r2, n, zeu = xr.apply_ufunc(
            bgc.euphotic_depth,
            par,
            depth,
            input_core_dims=[['PRES'], ['PRES']],  
            output_core_dims=[[], [], [], []],      
            vectorize=True,
            dask='parallelized',
            output_dtypes=[float, float, float, float],
            kwargs={'initial_fit_depth': 100, 'max_fit_depth': 186, 'step': 20})
        
        xds = xds.assign(
            KD_PAR=slope,
            KD_PAR_R2=r2,
            ZEU=zeu)
    
        # Chlorophyll fluorescence quenching
        xds["SOLAR_ELEVATION"] = bgc.calculate_solar_elevation(xds)

        ds = xds.sel(PRES=slice(0, 199))
        result = xr.apply_ufunc(bgc.apply_xing_quenching_correction,
                                ds['CHLA_FLUORESCENCE'],
                                ds['BBP_PHY'],
                                ds['DOWNWELLING_PAR'],
                                ds['MLD'],
                                ds['SOLAR_ELEVATION'],
                                input_core_dims=[['PRES'], ['PRES'], ['PRES'], [], []],
                                output_core_dims=[['PRES'], ['PRES'], [], []],
                                vectorize=True,
                                dask='parallelized',
                                output_dtypes=[float, float, float, bool])
    
        xds = xds.assign(CHLA= result[0],
                         NPQ=result[1],
                         QUENCHING_DEPTH=result[2],
                         QUENCHING_FLAG=result[3])
        
        result = xr.apply_ufunc(bgc.compute_fluorescence_to_chla_factor,
                             xds['DOWN_IRRADIANCE490'],
                             xds['CHLA'],
                             xds['SOLAR_ELEVATION'],
                             input_core_dims=[["PRES"], ["PRES"], []],
                             output_core_dims=[[], [], [], [], ["PRES"], ["PRES"]],
                             vectorize=True,
                             dask="parallelized",
                             output_dtypes=[float, float, float, float, float, float],)

        xds['CHLA_CONVERSION_FACTOR'] = result[0]
        xds['CHLA_CONVERSION_FACTOR_RSQ'] = result[3]
        xds['CHLA_CF_CN'] = result[4]
        xds['CHLA_CF_AN'] = result[5]
    
        # ---------------------------------------------------------
        # Replace missing conversion factors with float-wide median
        # ---------------------------------------------------------
        cf = xds["CHLA_CONVERSION_FACTOR"].values.astype(float)
        
        if np.all(np.isnan(cf)):
            name = os.path.basename(file)
            print(f"{name}: no valid conversion factors.")
            continue  # Skip float entirely if no valid conversion factors
        
        # Compute robust central tendency
        median = np.nanmedian(cf)
        mad = median_abs_deviation(cf, nan_policy="omit")
        
        # Outlier filtering using modified z-score
        if mad == 0 or np.isnan(mad):
            mask = np.isfinite(cf)
        else:
            z = np.abs((cf - median) / (1.4826 * mad))
            mask = np.isfinite(cf) & (z < 3)
        
        cf_filtered = np.where(mask, cf, np.nan)
        
        xds["CHLA_CF_FILTERED"] = (("N_PROF",), cf_filtered)
        
        mask_da = xr.DataArray(mask, dims=("N_PROF",), coords={"N_PROF": xds["N_PROF"]})
        
        xds["CHLA_CF_FILTERED_RSQ"] = xds["CHLA_CONVERSION_FACTOR_RSQ"].where(mask_da)
        
        # Replace missing/outlier values with float-wide median of filtered CF
        median_cf = np.nanmedian(cf_filtered)
        
        if np.isnan(median_cf):
            name = os.path.basename(file)
            print(f"{name}: no valid conversion factors after filtering.")
            continue  # Safety: all values were filtered out
        
        xds["CHLA_CF_FINAL"] = xds["CHLA_CF_FILTERED"].fillna(median_cf)
    
        # Apply conversion factor to CHLA
        xds["CHLA"] = xds["CHLA"] * xds["CHLA_CF_FINAL"]
    
        # Optional: Apply range filter to remove spurious CHLA
        xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

        # Calculate daily PAR from instantaneous PAR using a clear sky model
        xds["PAR_DAILY"] = bgc.compute_daily_par_clear_sky(xds, dt_seconds=60, par_top_depth=10)
        
        # Apply NPP algorithms
        ds = xds.sel(PRES=slice(0, 199))
        result = xr.apply_ufunc(bgc.cbpm_argo,
                                ds['CHLA'],
                                ds['CPHYTO'],
                                ds['PAR_DAILY'],
                                ds['DATETIME'].dt.dayofyear,
                                ds['LATITUDE'],
                                input_core_dims=[['PRES'], ['PRES'], [], [], []],
                                output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                                vectorize=True,
                                dask='parallelized',
                                output_dtypes=[float] * 4)
    
        xds = xds.assign(WESTBERRY_CBPM=result[0],
                         WESTBERRY_MU=result[1],
                         WESTBERRY_NUT_TEMP_FUNC=result[2],
                         WESTBERRY_IG_FUNC=result[3])
    
        xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] >= 0, np.nan)
    
        var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM']
            
        # Integrate top 200 m
        int_all = bgc.integrate_along_pres_multiple_vars(
            xds.where(xds.PRES <= 199, drop=False), var_list
        ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Integrate to Zeu
        zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
        int_zeu = bgc.integrate_along_pres_multiple_vars(
            xds.where(zeu_mask, drop=False), var_list
        ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Integrate to MLD
        mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
        int_mld = bgc.integrate_along_pres_multiple_vars(
            xds.where(mld_mask, drop=False), var_list
        ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
        
        # Merge safely
        int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')
    
        data_eu = xds[['WESTBERRY_CBPM', 
                       'WESTBERRY_IG_FUNC', 
                       'WESTBERRY_NUT_TEMP_FUNC', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
        int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
        int_all['ZEU_WESTBERRY_IG_FUNC'] = data_eu['WESTBERRY_IG_FUNC']
        int_all['ZEU_WESTBERRY_NUT_TEMP_FUNC'] = data_eu['WESTBERRY_NUT_TEMP_FUNC']
        int_all['ZEU_CHLA'] = data_eu['CHLA']
        int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
        int_all['ZEU_TEMP'] = data_eu['TEMP']
        int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']
    
        data_mld = xds[['WESTBERRY_CBPM', 
                       'WESTBERRY_IG_FUNC', 
                       'WESTBERRY_NUT_TEMP_FUNC', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
        int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
        int_all['MLD_WESTBERRY_IG_FUNC'] = data_mld['WESTBERRY_IG_FUNC']
        int_all['MLD_WESTBERRY_NUT_TEMP_FUNC'] = data_mld['WESTBERRY_NUT_TEMP_FUNC']
        int_all['MLD_CHLA'] = data_mld['CHLA']
        int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
        int_all['MLD_TEMP'] = data_mld['TEMP']
        int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']
    
        od = abs(1 / xds.KD_PAR)
        data_od = xds[['WESTBERRY_CBPM', 
                       'WESTBERRY_IG_FUNC', 
                       'WESTBERRY_NUT_TEMP_FUNC', 
                       'CHLA', 
                       'CPHYTO', 
                       'TEMP', 
                       'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
        int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
        int_all['OD1_WESTBERRY_IG_FUNC'] = data_od['WESTBERRY_IG_FUNC']
        int_all['OD1_WESTBERRY_NUT_TEMP_FUNC'] = data_od['WESTBERRY_NUT_TEMP_FUNC']
        int_all['OD1_CHLA'] = data_od['CHLA']
        int_all['OD1_CPHYTO'] = data_od['CPHYTO']
        int_all['OD1_TEMP'] = data_od['TEMP']
        int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
        
        int_all['DATETIME'] = xds.DATETIME
        int_all['LATITUDE'] = xds.LATITUDE
        int_all['LONGITUDE'] = xds.LONGITUDE
        int_all['ZEU'] = xds.ZEU
        int_all['MLD'] = xds.MLD
        int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
        int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG
    
        if int_all.N_PROF.shape[0] == 0:
            continue
    
        xds['BIOME'] = biomes.interp(lat=xds['LATITUDE'], lon=xds['LONGITUDE'], method='nearest')
        
        int_all['BIOME'] = biomes.interp(lat=int_all['LATITUDE'], lon=int_all['LONGITUDE'], method='nearest')
    
        xds = xds.drop_vars(['lat', 'lon'])
    
        int_all = int_all.drop_vars(['lat', 'lon'])
    
        xds.to_netcdf(
            f"{outdir}/{name}.nc",
            format="NETCDF4",
            encoding=bgc.make_encoding(xds, complevel=1),
        )
        
        int_all.to_netcdf(
            f"{outdir_npp}/{name}_NPP.nc",
            format="NETCDF4",
            encoding=bgc.make_encoding(int_all, complevel=1),
        )

  0%|▏                                                                                  | 1/386 [00:01<07:21,  1.15s/it]

1902651_Sprof.nc: no valid conversion factors.


  3%|██                                                                                | 10/386 [00:52<29:52,  4.77s/it]

5907054_Sprof.nc: no valid conversion factors.


  4%|███▏                                                                              | 15/386 [01:00<15:42,  2.54s/it]

...

4903619_Sprof.nc: no valid conversion factors.


100%|██████████████████████████████████████████████████████████████████████████████▊| 385/386 [1:04:00<00:05,  5.81s/it]

4902690_Sprof.nc: no valid conversion factors.


100%|███████████████████████████████████████████████████████████████████████████████| 386/386 [1:04:01<00:00,  9.95s/it]

4902691_Sprof.nc: no valid conversion factors.


### SOCA - Floats with Irradiance Sensors

In [9]:
pending_file = Path("../DATA/flist_with_par_pending.txt")

if not pending_file.exists():
    pending_file.write_text("\n".join(flist_with_par), encoding="utf-8")

flist_with_par = [
    line.strip()
    for line in pending_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

skipped_files = []

SKIP_INACTIVE_FLOATS = True
INACTIVE_FLOAT_DAYS = 365

outdir = Path(
    "../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITH_IRRADIANCE"
)
outdir_npp = Path(
    "../DATA/Output Final/NPP_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITH_IRRADIANCE"
)

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

with tqdm(total=len(flist_with_par), desc="Processing files", dynamic_ncols=True) as pbar:

    for file_i, file in enumerate(flist_with_par, start=1):

        file_name = Path(file).name
        name = os.path.basename(file).split(".")[0]

        final_profile_file = outdir / f"{name}.nc"
        final_npp_file = outdir_npp / f"{name}_NPP.nc"

        tmp_profile_file = outdir / f"{name}.tmp.nc"
        tmp_npp_file = outdir_npp / f"{name}_NPP.tmp.nc"

        pbar.set_description(f"File {file_i}/{len(flist_with_par)}")
        pbar.set_postfix_str(file_name)

        try:

            base_vars = [
                "PRES", "PRES_QC",
                "TEMP", "TEMP_QC", "TEMP_ADJUSTED", "TEMP_ADJUSTED_QC",
                "PSAL", "PSAL_QC", "PSAL_ADJUSTED", "PSAL_ADJUSTED_QC",
                "BBP700", "BBP700_QC", "BBP700_ADJUSTED", "BBP700_ADJUSTED_QC",
                "CHLA", "CHLA_QC",
                "CHLA_FLUORESCENCE", "CHLA_FLUORESCENCE_QC",
                "CHLA_FLUORESCENCE_ADJUSTED", "CHLA_FLUORESCENCE_ADJUSTED_QC",
            ]

            xds = bgc.process_bgc_argo_file(file, base_vars=base_vars)

            if xds is None:
                print(f"⚠️ Skipping {file} — dataset is None.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds = bgc.clean_range_filter(xds, "BBP700", 0, 0.05)

            xds["BBP470"] = xds["BBP700"] * (470 / 700) ** -0.78

            xds = bgc.calculate_bbp_nap_and_phyto(
                xds,
                bbp_var="BBP470",
                depth_var="PRES",
            )

            if xds["BBP_PHY"].isnull().all():
                print(f"{name}: no valid BBP_PHY after QC/filtering – skipping.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["CPHYTO"] = 12128 * xds["BBP_PHY"]
            xds["CPHYTO"] = xds["CPHYTO"].where(
                np.isfinite(xds["CPHYTO"]) & (xds["CPHYTO"] >= 0)
            )

            required_meta = ["JULD", "LATITUDE", "LONGITUDE"]
            missing_meta = [m for m in required_meta if m not in xds]

            if missing_meta:
                print(f"{name}: missing metadata fields {missing_meta}; skipping.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds = xds.dropna(dim="N_PROF", subset=required_meta)
            xds = xds.dropna(dim="N_PROF", how="all", subset=xds.data_vars)

            if xds.sizes.get("N_PROF", 0) <= 1:
                print(f"{name}: fails final QC – not enough valid profiles.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            profile_times = pd.to_datetime(xds.DATETIME.values)

            valid_mask = (
                (profile_times >= DATA_START + pd.Timedelta(days=5))
                & (profile_times <= DATA_END - pd.Timedelta(days=5))
            )

            xds = xds.isel(N_PROF=valid_mask)

            if xds.sizes.get("N_PROF", 0) <= 1:
                print(f"{name}: no valid profiles within satellite overlap window.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["PSAL_ABSOLUTE"] = gsw.SA_from_SP(
                xds.PSAL,
                xds.PRES,
                xds.LONGITUDE,
                xds.LATITUDE,
            )

            xds["DENSITY"] = gsw.pot_rho_t_exact(
                xds.PSAL_ABSOLUTE,
                xds.TEMP,
                xds.PRES,
                0,
            )

            xds["MLD"] = bgc.compute_mld_density_threshold(
                xds,
                rho_name="DENSITY",
                depth_name="PRES",
                ref_depth=10,
                delta_rho=0.03,
            )

            # ------------------------------------------------------------
            # Skip inactive floats before doing unnecessary update work
            # ------------------------------------------------------------
            skip_float, latest_profile_time, latest_profile_str, lookup_cutoff = (
                bgc.should_skip_inactive_float(
                    xds,
                    skip_inactive=SKIP_INACTIVE_FLOATS,
                    inactive_days=INACTIVE_FLOAT_DAYS,
                )
            )

            if skip_float:
                print(
                    f"{name}: latest profile {latest_profile_str} is older than "
                    f"{INACTIVE_FLOAT_DAYS} days — skipping inactive float."
                )

                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)

                try:
                    xds.close()
                except Exception:
                    pass

                del xds
                gc.collect()
                continue

            # ------------------------------------------------------------
            # Reuse old SAT_* values from copied archive where possible
            # ------------------------------------------------------------
            xds, lookup_idx = bgc.initialise_or_carry_forward_sat_vars(
                xds,
                old_profile_file=final_profile_file,
            )

            # ------------------------------------------------------------
            # Restrict new satellite lookups to genuinely recent profiles only
            # ------------------------------------------------------------
            n_missing_sat = len(lookup_idx)
            do_satellite_lookup = False

            if n_missing_sat == 0:
                print(f"{name}: no missing satellite values — skipping satellite lookup.")

            else:
                profile_time = pd.to_datetime(xds["DATETIME"].values)

                recent_profile_mask = (
                    pd.notna(profile_time)
                    & (profile_time >= lookup_cutoff)
                )

                lookup_idx = [
                    idx for idx in lookup_idx
                    if recent_profile_mask[idx]
                ]

                print(
                    f"{name}: {n_missing_sat} missing SAT profiles before date filtering; "
                    f"{len(lookup_idx)} within last {INACTIVE_FLOAT_DAYS} days."
                )

                if len(lookup_idx) == 0:
                    print(
                        f"{name}: all missing SAT profiles are older than "
                        f"{INACTIVE_FLOAT_DAYS} days — skipping satellite lookup."
                    )
                else:
                    do_satellite_lookup = True

            # ------------------------------------------------------------
            # PAR lookup only for missing/new/recent profiles
            # ------------------------------------------------------------
            if do_satellite_lookup:
                dates = xds.DATETIME.values
                lat = xds.LATITUDE.values
                lon = xds.LONGITUDE.values

                for jj, n in enumerate(lookup_idx, start=1):

                    if jj % 10 == 0:
                        pbar.set_postfix_str(
                            f"{file_name} | PAR new {jj}/{len(lookup_idx)}"
                        )

                    dt = dates[n]
                    lat_val = lat[n]
                    lon_val = lon[n]

                    if np.isnan(lat_val) or np.isnan(lon_val) or pd.isnull(dt):
                        xds["SAT_PAR"].values[n] = np.nan
                        continue

                    target_date = dt.astype("datetime64[D]")

                    abs_diff = (
                        np.abs(sorted_par_dates - target_date)
                        .astype("timedelta64[D]")
                        .astype(int)
                    )

                    closest_index = np.argmin(abs_diff)

                    start_idx = max(0, closest_index - 5)
                    end_idx = min(len(sorted_par_files), closest_index + 6)

                    selected_files = sorted_par_files[start_idx:end_idx]
                    selected_dates = sorted_par_dates[start_idx:end_idx]

                    ds = None
                    par_data = None

                    try:
                        ds = xr.open_mfdataset(
                            selected_files,
                            concat_dim="time",
                            combine="nested",
                            chunks={},
                        )

                        ds["time"] = selected_dates.astype("datetime64[ns]")
                        ds = ds.sortby("time")

                        lat_idx = np.argmin(abs(ds.lat.values - lat_val))
                        lon_idx = np.argmin(abs(ds.lon.values - lon_val))

                        par_data = ds.isel(
                            lat=slice(
                                max(lat_idx - 2, 0),
                                min(lat_idx + 3, ds.sizes["lat"]),
                            ),
                            lon=slice(
                                max(lon_idx - 2, 0),
                                min(lon_idx + 3, ds.sizes["lon"]),
                            ),
                        )

                        par_value = bgc.get_nearest_valid_value(
                            par_data,
                            "PAR_mean",
                            lat_val,
                            lon_val,
                            dt,
                        )

                        xds["SAT_PAR"].values[n] = par_value

                    except Exception as e:
                        print(f"{name}: PAR lookup failed for profile {n}: {e}")
                        xds["SAT_PAR"].values[n] = np.nan

                    finally:
                        try:
                            if ds is not None:
                                ds.close()
                        except Exception:
                            pass

                        try:
                            del ds, par_data
                        except Exception:
                            pass

            # ------------------------------------------------------------
            # RRS/KD lookup only for missing/new profiles
            # ------------------------------------------------------------
            profile_lats = xds.LATITUDE.values.astype(float)
            profile_lons = xds.LONGITUDE.values.astype(float)
            profile_lons = ((profile_lons + 180) % 360) - 180
            profile_times = pd.to_datetime(xds.DATETIME.values)

            variables = ["RRS412", "RRS443", "RRS490", "RRS555", "RRS670"]

            lookup_idx = np.asarray(lookup_idx, dtype=int)
            profile_months = profile_times.to_period("M")

            if lookup_idx.size > 0:
                unique_months = np.sort(profile_months[lookup_idx].unique())
            else:
                unique_months = []

            dx = 0.036

            RRS_list = [
                {
                    var: (
                        float(xds[f"SAT_{var}"].values[i])
                        if np.isfinite(xds[f"SAT_{var}"].values[i])
                        else np.nan
                    )
                    for var in variables
                }
                for i in range(len(profile_lats))
            ]

            KD_list = list(xds["SAT_KD490"].values.astype(np.float32))

            LAT_MIN, LAT_MAX = -89.97917, 89.97917
            LON_MIN, LON_MAX = -179.97917, 179.97917

            for month_i, month in enumerate(unique_months, start=1):

                month_lookup_mask = profile_months[lookup_idx] == month
                month_idx = lookup_idx[month_lookup_mask]

                if month_idx.size == 0:
                    continue

                pbar.set_description(
                    f"File {file_i}/{len(flist_with_par)} | "
                    f"Lookup month {month_i}/{len(unique_months)}"
                )
                pbar.set_postfix_str(
                    f"{file_name} | {month} | {len(month_idx)} profiles"
                )

                lat_min = float(np.nanmin(profile_lats[month_idx]) - 2 * dx)
                lat_max = float(np.nanmax(profile_lats[month_idx]) + 2 * dx)
                lon_min = float(np.nanmin(profile_lons[month_idx]) - 2 * dx)
                lon_max = float(np.nanmax(profile_lons[month_idx]) + 2 * dx)

                lat_min = float(np.clip(lat_min, LAT_MIN, LAT_MAX))
                lat_max = float(np.clip(lat_max, LAT_MIN, LAT_MAX))
                lon_min = float(np.clip(lon_min, LON_MIN, LON_MAX))
                lon_max = float(np.clip(lon_max, LON_MIN, LON_MAX))

                time_min = profile_times[month_idx].min() - pd.Timedelta(days=5)
                time_max = profile_times[month_idx].max() + pd.Timedelta(days=5)

                regional_rrs = copernicusmarine.open_dataset(
                    dataset_id="cmems_obs-oc_glo_bgc-reflectance_my_l3-multi-4km_P1D",
                    start_datetime=str(time_min),
                    end_datetime=str(time_max),
                    minimum_latitude=lat_min,
                    maximum_latitude=lat_max,
                    minimum_longitude=lon_min,
                    maximum_longitude=lon_max,
                    variables=variables,
                ).load()

                regional_kd = copernicusmarine.open_dataset(
                    dataset_id="cmems_obs-oc_glo_bgc-transp_my_l3-multi-4km_P1D",
                    start_datetime=str(time_min),
                    end_datetime=str(time_max),
                    minimum_latitude=lat_min,
                    maximum_latitude=lat_max,
                    minimum_longitude=lon_min,
                    maximum_longitude=lon_max,
                    variables=["KD490"],
                ).load()

                rrs_lats = regional_rrs.latitude.values
                rrs_lons = regional_rrs.longitude.values

                for n in month_idx:

                    prof_lat = float(profile_lats[n])
                    prof_lon = float(profile_lons[n])
                    prof_time = np.datetime64(profile_times[n])

                    lat_idx = np.abs(rrs_lats - prof_lat).argmin()
                    lon_idx = np.abs(rrs_lons - prof_lon).argmin()

                    lat_slice = slice(
                        max(lat_idx - 2, 0),
                        min(lat_idx + 3, regional_rrs.sizes["latitude"]),
                    )
                    lon_slice = slice(
                        max(lon_idx - 2, 0),
                        min(lon_idx + 3, regional_rrs.sizes["longitude"]),
                    )
                    time_slice = slice(
                        prof_time - np.timedelta64(5, "D"),
                        prof_time + np.timedelta64(5, "D"),
                    )

                    sub_lat_vals = rrs_lats[lat_slice]
                    sub_lon_vals = rrs_lons[lon_slice]

                    cube_lats, cube_lons = np.meshgrid(
                        sub_lat_vals,
                        sub_lon_vals,
                        indexing="ij",
                    )

                    this_rrs = {}

                    for var in variables:
                        value = bgc.get_nearest_valid_value_cube(
                            regional_rrs,
                            var,
                            prof_lat,
                            prof_lon,
                            prof_time,
                            lat_slice,
                            lon_slice,
                            time_slice,
                            cube_lats,
                            cube_lons,
                        )

                        this_rrs[var] = value * np.pi if np.isfinite(value) else np.nan

                    this_kd = bgc.get_nearest_valid_value_cube(
                        regional_kd,
                        "KD490",
                        prof_lat,
                        prof_lon,
                        prof_time,
                        lat_slice,
                        lon_slice,
                        time_slice,
                        cube_lats,
                        cube_lons,
                    )

                    RRS_list[n] = this_rrs
                    KD_list[n] = this_kd

                regional_rrs.close()
                regional_kd.close()

                del regional_rrs, regional_kd
                gc.collect()

            RRS_df = pd.DataFrame(RRS_list)

            for var in variables:
                xds[f"SAT_{var}"] = xr.DataArray(
                    RRS_df[var].values,
                    dims="N_PROF",
                    coords={"N_PROF": xds.N_PROF},
                )

            xds["SAT_KD490"] = xr.DataArray(
                np.array(KD_list, dtype=np.float32),
                dims="N_PROF",
                coords={"N_PROF": xds.N_PROF},
            )

            # ------------------------------------------------------------
            # Irradiance profile reconstruction
            # ------------------------------------------------------------
            PAR_DATA = []
            ED380_DATA = []
            ED412_DATA = []
            ED490_DATA = []

            depth_output = np.linspace(0, 250, 51)

            for i in range(len(xds.N_PROF)):

                try:
                    LAT = xds.LATITUDE[i].values
                    LON = xds.LONGITUDE[i].values

                    timezone_str = tf.certain_timezone_at(lat=LAT, lng=LON)

                    if timezone_str is None:
                        raise ValueError(
                            f"Unable to find timezone for LAT={LAT}, LON={LON}"
                        )

                    timezone = pytz.timezone(timezone_str)

                    LC = pd.to_datetime(xds.DATETIME[i].values) + timezone.utcoffset(
                        pd.to_datetime(xds.DATETIME[i].values)
                    )

                    LC_Hour_Deci = LC.hour + (LC.minute / 60)

                    PAR = np.array(xds.SAT_PAR[i].values, dtype=np.float32)
                    MLD = np.array(xds.MLD[i].values, dtype=np.float32)
                    ZPD = 1 / np.array(xds.SAT_KD490[i].values, dtype=np.float32)

                    RHO412 = np.array(xds.SAT_RRS412[i].values, dtype=np.float32)
                    RHO443 = np.array(xds.SAT_RRS443[i].values, dtype=np.float32)
                    RHO490 = np.array(xds.SAT_RRS490[i].values, dtype=np.float32)
                    RHO555 = np.array(xds.SAT_RRS555[i].values, dtype=np.float32)
                    RHO670 = np.array(xds.SAT_RRS670[i].values, dtype=np.float32)

                    doy = np.array(LC.timetuple().tm_yday, dtype=np.float64)

                    sin_doy = np.sin(rad_date(doy))
                    cos_doy = np.cos(rad_date(doy))
                    sin_LC_hour = np.sin(rad_LC_hour(LC_Hour_Deci))
                    cos_LC_hour = np.cos(rad_LC_hour(LC_Hour_Deci))

                    temp = np.array(xds.TEMP[i, :250].values, dtype=np.float64)
                    sal = np.array(xds.PSAL[i, :250].values, dtype=np.float64)
                    pres = xds.PRES[:250].values

                    valid = (~np.isnan(pres)) & (~np.isnan(sal)) & (~np.isnan(temp))

                    pres2 = pres[valid]
                    temp2 = temp[valid]
                    sal2 = sal[valid]

                    pres3 = pres2[pres2 <= 250]
                    pres4 = pres2[pres2 <= 50]

                    if (len(pres3) >= 15) & (len(pres4) >= 5):

                        non_array_inputs = np.array(
                            [
                                PAR,
                                MLD,
                                ZPD,
                                RHO412,
                                RHO443,
                                RHO490,
                                RHO555,
                                RHO670,
                                sin_doy,
                                cos_doy,
                                sin_LC_hour,
                                cos_LC_hour,
                            ],
                            dtype=np.float64,
                        )

                        if any(np.isnan(x).any() for x in non_array_inputs):

                            PAR_ADJUSTED = np.full(51, np.nan)
                            ED380_ADJUSTED = np.full(51, np.nan)
                            ED412_ADJUSTED = np.full(51, np.nan)
                            ED490_ADJUSTED = np.full(51, np.nan)

                        else:

                            PAR_ADJUSTED = PAR_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED380_ADJUSTED = ED380_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED412_ADJUSTED = ED412_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED490_ADJUSTED = ED490_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                    else:
                        PAR_ADJUSTED = np.full(51, np.nan)
                        ED380_ADJUSTED = np.full(51, np.nan)
                        ED412_ADJUSTED = np.full(51, np.nan)
                        ED490_ADJUSTED = np.full(51, np.nan)

                    PAR_DATA.append(PAR_ADJUSTED)
                    ED380_DATA.append(ED380_ADJUSTED)
                    ED412_DATA.append(ED412_ADJUSTED)
                    ED490_DATA.append(ED490_ADJUSTED)

                except Exception as e:
                    print(f"Error processing profile {i}: {e}")

                    PAR_DATA.append(np.full(51, np.nan))
                    ED380_DATA.append(np.full(51, np.nan))
                    ED412_DATA.append(np.full(51, np.nan))
                    ED490_DATA.append(np.full(51, np.nan))

            PAR_DATA = pd.DataFrame(np.array(PAR_DATA).T)
            ED380_DATA = pd.DataFrame(np.array(ED380_DATA).T)
            ED412_DATA = pd.DataFrame(np.array(ED412_DATA).T)
            ED490_DATA = pd.DataFrame(np.array(ED490_DATA).T)

            depths = np.linspace(0, 250, 51)

            PAR_DATA.index = depths
            ED380_DATA.index = depths
            ED412_DATA.index = depths
            ED490_DATA.index = depths

            PAR_DATA = PAR_DATA.reindex(xds.PRES.values).interpolate()
            ED380_DATA = ED380_DATA.reindex(xds.PRES.values).interpolate()
            ED412_DATA = ED412_DATA.reindex(xds.PRES.values).interpolate()
            ED490_DATA = ED490_DATA.reindex(xds.PRES.values).interpolate()

            xds["DOWNWELLING_PAR"] = xr.DataArray(
                np.array(PAR_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds = bgc.clean_range_filter(xds, "DOWNWELLING_PAR", 0.001, 4672)

            xds["DOWN_IRRADIANCE380"] = xr.DataArray(
                np.array(ED380_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds["DOWN_IRRADIANCE412"] = xr.DataArray(
                np.array(ED412_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds["DOWN_IRRADIANCE490"] = xr.DataArray(
                np.array(ED490_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE380", 1e-5, 1.7)
            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE412", 1e-5, 2.9)
            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE490", 1e-5, 3.4)

            # ------------------------------------------------------------
            # Euphotic depth
            # ------------------------------------------------------------
            par = xds["DOWNWELLING_PAR"].sel(PRES=slice(0, 199))
            depth = xds["PRES"].sel(PRES=slice(0, 199))

            slope, r2, n, zeu = xr.apply_ufunc(
                bgc.euphotic_depth,
                par,
                depth,
                input_core_dims=[["PRES"], ["PRES"]],
                output_core_dims=[[], [], [], []],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float, float, float],
                kwargs={
                    "initial_fit_depth": 100,
                    "max_fit_depth": 186,
                    "step": 20,
                },
            )

            xds = xds.assign(
                KD_PAR=slope,
                KD_PAR_R2=r2,
                ZEU=zeu,
            )

            # ------------------------------------------------------------
            # Chlorophyll fluorescence quenching
            # ------------------------------------------------------------
            xds["SOLAR_ELEVATION"] = bgc.calculate_solar_elevation(xds)

            ds = xds.sel(PRES=slice(0, 199))

            result = xr.apply_ufunc(
                bgc.apply_xing_quenching_correction,
                ds["CHLA_FLUORESCENCE"],
                ds["BBP_PHY"],
                ds["DOWNWELLING_PAR"],
                ds["MLD"],
                ds["SOLAR_ELEVATION"],
                ds["PRES"],
                input_core_dims=[
                    ["PRES"],
                    ["PRES"],
                    ["PRES"],
                    [],
                    [],
                    ["PRES"],
                ],
                output_core_dims=[["PRES"], ["PRES"], [], []],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float, float, bool],
            )

            xds = xds.assign(
                CHLA=result[0],
                NPQ=result[1],
                QUENCHING_DEPTH=result[2],
                QUENCHING_FLAG=result[3],
            )

            xds["CHLA"] = xds.CHLA / 2
            xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

            # ------------------------------------------------------------
            # NPP
            # ------------------------------------------------------------
            xds["PAR_DAILY"] = bgc.compute_daily_par_clear_sky(xds, dt_seconds=60, par_top_depth=10)
            ds = xds.sel(PRES=slice(0, 199))

            result = xr.apply_ufunc(
                bgc.cbpm_argo,
                ds["CHLA"],
                ds["CPHYTO"],
                ds["PAR_DAILY"],
                ds["DATETIME"].dt.dayofyear,
                ds["LATITUDE"],
                input_core_dims=[["PRES"], ["PRES"], [], [], []],
                output_core_dims=[["PRES"], ["PRES"], ["PRES"], ["PRES"]],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float] * 4,
            )

            xds = xds.assign(
                WESTBERRY_CBPM=result[0],
                WESTBERRY_MU=result[1],
                WESTBERRY_NUT_TEMP_FUNC=result[2],
                WESTBERRY_IG_FUNC=result[3],
            )

            xds["WESTBERRY_CBPM"] = xds["WESTBERRY_CBPM"].where(
                xds["WESTBERRY_CBPM"] >= 0,
                np.nan,
            )

            var_list = ["CHLA", "CPHYTO", "WESTBERRY_CBPM"]

            int_all = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(xds.PRES <= 199, drop=False),
                    var_list,
                )
                .rename({v: f"{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)

            int_zeu = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(zeu_mask, drop=False),
                    var_list,
                )
                .rename({v: f"ZEU_{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)

            int_mld = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(mld_mask, drop=False),
                    var_list,
                )
                .rename({v: f"MLD_{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            int_all = xr.merge([int_all, int_zeu, int_mld], join="outer")

            mean_vars = [
                "WESTBERRY_CBPM",
                "WESTBERRY_IG_FUNC",
                "WESTBERRY_NUT_TEMP_FUNC",
                "CHLA",
                "CPHYTO",
                "TEMP",
                "DOWNWELLING_PAR",
            ]

            data_eu = xds[mean_vars].where(
                xds.PRES <= xds.ZEU,
                drop=False,
            ).mean(dim="PRES")

            int_all["ZEU_WESTBERRY_CBPM"] = data_eu["WESTBERRY_CBPM"]
            int_all["ZEU_WESTBERRY_IG_FUNC"] = data_eu["WESTBERRY_IG_FUNC"]
            int_all["ZEU_WESTBERRY_NUT_TEMP_FUNC"] = data_eu["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["ZEU_CHLA"] = data_eu["CHLA"]
            int_all["ZEU_CPHYTO"] = data_eu["CPHYTO"]
            int_all["ZEU_TEMP"] = data_eu["TEMP"]
            int_all["ZEU_DOWNWELLING_PAR"] = data_eu["DOWNWELLING_PAR"]

            data_mld = xds[mean_vars].where(
                xds.PRES <= xds.MLD,
                drop=False,
            ).mean(dim="PRES")

            int_all["MLD_WESTBERRY_CBPM"] = data_mld["WESTBERRY_CBPM"]
            int_all["MLD_WESTBERRY_IG_FUNC"] = data_mld["WESTBERRY_IG_FUNC"]
            int_all["MLD_WESTBERRY_NUT_TEMP_FUNC"] = data_mld["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["MLD_CHLA"] = data_mld["CHLA"]
            int_all["MLD_CPHYTO"] = data_mld["CPHYTO"]
            int_all["MLD_TEMP"] = data_mld["TEMP"]
            int_all["MLD_DOWNWELLING_PAR"] = data_mld["DOWNWELLING_PAR"]

            od = abs(1 / xds.KD_PAR)

            data_od = xds[mean_vars].where(
                xds.PRES <= od,
                drop=False,
            ).mean(dim="PRES")

            int_all["OD1_WESTBERRY_CBPM"] = data_od["WESTBERRY_CBPM"]
            int_all["OD1_WESTBERRY_IG_FUNC"] = data_od["WESTBERRY_IG_FUNC"]
            int_all["OD1_WESTBERRY_NUT_TEMP_FUNC"] = data_od["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["OD1_CHLA"] = data_od["CHLA"]
            int_all["OD1_CPHYTO"] = data_od["CPHYTO"]
            int_all["OD1_TEMP"] = data_od["TEMP"]
            int_all["OD1_DOWNWELLING_PAR"] = data_od["DOWNWELLING_PAR"]

            int_all["DATETIME"] = xds.DATETIME
            int_all["LATITUDE"] = xds.LATITUDE
            int_all["LONGITUDE"] = xds.LONGITUDE
            int_all["ZEU"] = xds.ZEU
            int_all["MLD"] = xds.MLD
            int_all["SOLAR_ELEVATION"] = xds.SOLAR_ELEVATION
            int_all["QUENCHING_FLAG"] = xds.QUENCHING_FLAG

            if int_all.N_PROF.shape[0] == 0:
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["BIOME"] = biomes.interp(
                lat=xds["LATITUDE"],
                lon=xds["LONGITUDE"],
                method="nearest",
            )

            int_all["BIOME"] = biomes.interp(
                lat=int_all["LATITUDE"],
                lon=int_all["LONGITUDE"],
                method="nearest",
            )

            drop_from_xds = [v for v in ["lat", "lon"] if v in xds]
            drop_from_int = [v for v in ["lat", "lon"] if v in int_all]

            if drop_from_xds:
                xds = xds.drop_vars(drop_from_xds)

            if drop_from_int:
                int_all = int_all.drop_vars(drop_from_int)

            xds.to_netcdf(
                tmp_profile_file,
                format="NETCDF4",
                encoding=bgc.make_encoding(xds, complevel=1),
            )

            int_all.to_netcdf(
                tmp_npp_file,
                format="NETCDF4",
                encoding=bgc.make_encoding(int_all, complevel=1),
            )

            tmp_profile_file.replace(final_profile_file)
            tmp_npp_file.replace(final_npp_file)

            bgc.remove_from_pending(pending_file, file)

            pbar.update(1)

            try:
                xds.close()
                int_all.close()
            except Exception:
                pass

            del xds, int_all
            gc.collect()

        except Exception as e:
            print(f"❌ Error processing {name}: {e}")
            skipped_files.append(name)
            pbar.update(1)

            try:
                xds.close()
            except Exception:
                pass

            try:
                int_all.close()
            except Exception:
                pass

            gc.collect()
            continue

File 1/386:   0%|                                                             | 0/386 [00:00<?, ?it/s, 1902651_Sprof.nc]

1902651_Sprof: 78 profiles total; 2 need new satellite lookup


File 2/386:   0%|▏                                                  | 1/386 [00:41<4:28:23, 41.83s/it, 2903863_Sprof.nc]

2903863_Sprof: 197 profiles total; 4 need new satellite lookup


File 3/386:   1%|▎                                                  | 2/386 [01:51<6:12:39, 58.23s/it, 2903884_Sprof.nc]

...

File 385/386:  99%|████████████████████████████████████████████▊| 384/386 [12:41:49<05:38, 169.04s/it, 4902690_Sprof.nc]

4902690_Sprof: 34 profiles total; 7 need new satellite lookup


File 386/386: 100%|████████████████████████████████████████████▉| 385/386 [12:42:21<02:07, 127.96s/it, 4902691_Sprof.nc]

4902691_Sprof: 34 profiles total; 6 need new satellite lookup


File 386/386 | Lookup month 4/4: 100%|███| 386/386 [12:42:56<00:00, 118.59s/it, 4902691_Sprof.nc | 2026-04 | 2 profiles]


### SOCA - Floats without Irradiance Sensors

In [6]:
pending_file = Path("../DATA/flist_without_par_pending.txt")

if not pending_file.exists():
    pending_file.write_text("\n".join(flist_without_par), encoding="utf-8")

flist_without_par = [
    line.strip()
    for line in pending_file.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

skipped_files = []

SKIP_INACTIVE_FLOATS = True
INACTIVE_FLOAT_DAYS = 365

outdir = Path(
    "../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITHOUT_IRRADIANCE"
)
outdir_npp = Path(
    "../DATA/Output Final/NPP_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITHOUT_IRRADIANCE"
)

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

with tqdm(total=len(flist_without_par), desc="Processing files", dynamic_ncols=True) as pbar:

    for file_i, file in enumerate(flist_without_par, start=1):

        file_name = Path(file).name
        name = os.path.basename(file).split(".")[0]

        final_profile_file = outdir / f"{name}.nc"
        final_npp_file = outdir_npp / f"{name}_NPP.nc"

        tmp_profile_file = outdir / f"{name}.tmp.nc"
        tmp_npp_file = outdir_npp / f"{name}_NPP.tmp.nc"

        pbar.set_description(f"File {file_i}/{len(flist_without_par)}")
        pbar.set_postfix_str(file_name)

        try:

            base_vars = [
                "PRES", "PRES_QC",
                "TEMP", "TEMP_QC", "TEMP_ADJUSTED", "TEMP_ADJUSTED_QC",
                "PSAL", "PSAL_QC", "PSAL_ADJUSTED", "PSAL_ADJUSTED_QC",
                "BBP700", "BBP700_QC", "BBP700_ADJUSTED", "BBP700_ADJUSTED_QC",
                "CHLA", "CHLA_QC",
                "CHLA_FLUORESCENCE", "CHLA_FLUORESCENCE_QC",
                "CHLA_FLUORESCENCE_ADJUSTED", "CHLA_FLUORESCENCE_ADJUSTED_QC",
            ]

            xds = bgc.process_bgc_argo_file(file, base_vars=base_vars)

            if xds is None:
                print(f"⚠️ Skipping {file} — dataset is None.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds = bgc.clean_range_filter(xds, "BBP700", 0, 0.05)

            xds["BBP470"] = xds["BBP700"] * (470 / 700) ** -0.78

            xds = bgc.calculate_bbp_nap_and_phyto(
                xds,
                bbp_var="BBP470",
                depth_var="PRES",
            )

            if xds["BBP_PHY"].isnull().all():
                print(f"{name}: no valid BBP_PHY after QC/filtering – skipping.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["CPHYTO"] = 12128 * xds["BBP_PHY"]
            xds["CPHYTO"] = xds["CPHYTO"].where(
                np.isfinite(xds["CPHYTO"]) & (xds["CPHYTO"] >= 0)
            )

            required_meta = ["JULD", "LATITUDE", "LONGITUDE"]
            missing_meta = [m for m in required_meta if m not in xds]

            if missing_meta:
                print(f"{name}: missing metadata fields {missing_meta}; skipping.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds = xds.dropna(dim="N_PROF", subset=required_meta)
            xds = xds.dropna(dim="N_PROF", how="all", subset=xds.data_vars)

            if xds.sizes.get("N_PROF", 0) <= 1:
                print(f"{name}: fails final QC – not enough valid profiles.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            profile_times = pd.to_datetime(xds.DATETIME.values)

            valid_mask = (
                (profile_times >= DATA_START + pd.Timedelta(days=5))
                & (profile_times <= DATA_END - pd.Timedelta(days=5))
            )

            xds = xds.isel(N_PROF=valid_mask)

            if xds.sizes.get("N_PROF", 0) <= 1:
                print(f"{name}: no valid profiles within satellite overlap window.")
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["PSAL_ABSOLUTE"] = gsw.SA_from_SP(
                xds.PSAL,
                xds.PRES,
                xds.LONGITUDE,
                xds.LATITUDE,
            )

            xds["DENSITY"] = gsw.pot_rho_t_exact(
                xds.PSAL_ABSOLUTE,
                xds.TEMP,
                xds.PRES,
                0,
            )

            xds["MLD"] = bgc.compute_mld_density_threshold(
                xds,
                rho_name="DENSITY",
                depth_name="PRES",
                ref_depth=10,
                delta_rho=0.03,
            )

            # ------------------------------------------------------------
            # Skip inactive floats before doing unnecessary update work
            # ------------------------------------------------------------
            skip_float, latest_profile_time, latest_profile_str, lookup_cutoff = (
                bgc.should_skip_inactive_float(
                    xds,
                    skip_inactive=SKIP_INACTIVE_FLOATS,
                    inactive_days=INACTIVE_FLOAT_DAYS,
                )
            )

            if skip_float:
                print(
                    f"{name}: latest profile {latest_profile_str} is older than "
                    f"{INACTIVE_FLOAT_DAYS} days — skipping inactive float."
                )

                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)

                try:
                    xds.close()
                except Exception:
                    pass

                del xds
                gc.collect()
                continue

            # ------------------------------------------------------------
            # Reuse old SAT_* values from copied archive where possible
            # ------------------------------------------------------------
            xds, lookup_idx = bgc.initialise_or_carry_forward_sat_vars(
                xds,
                old_profile_file=final_profile_file,
            )

            # ------------------------------------------------------------
            # Restrict new satellite lookups to genuinely recent profiles only
            # ------------------------------------------------------------
            n_missing_sat = len(lookup_idx)
            do_satellite_lookup = False

            if n_missing_sat == 0:
                print(f"{name}: no missing satellite values — skipping satellite lookup.")

            else:
                profile_time = pd.to_datetime(xds["DATETIME"].values)

                recent_profile_mask = (
                    pd.notna(profile_time)
                    & (profile_time >= lookup_cutoff)
                )

                lookup_idx = [
                    idx for idx in lookup_idx
                    if recent_profile_mask[idx]
                ]

                print(
                    f"{name}: {n_missing_sat} missing SAT profiles before date filtering; "
                    f"{len(lookup_idx)} within last {INACTIVE_FLOAT_DAYS} days."
                )

                if len(lookup_idx) == 0:
                    print(
                        f"{name}: all missing SAT profiles are older than "
                        f"{INACTIVE_FLOAT_DAYS} days — skipping satellite lookup."
                    )
                else:
                    do_satellite_lookup = True

            # ------------------------------------------------------------
            # PAR lookup only for missing/new/recent profiles
            # ------------------------------------------------------------
            if do_satellite_lookup:
                dates = xds.DATETIME.values
                lat = xds.LATITUDE.values
                lon = xds.LONGITUDE.values

                for jj, n in enumerate(lookup_idx, start=1):

                    if jj % 10 == 0:
                        pbar.set_postfix_str(
                            f"{file_name} | PAR new {jj}/{len(lookup_idx)}"
                        )

                    dt = dates[n]
                    lat_val = lat[n]
                    lon_val = lon[n]

                    if np.isnan(lat_val) or np.isnan(lon_val) or pd.isnull(dt):
                        xds["SAT_PAR"].values[n] = np.nan
                        continue

                    target_date = dt.astype("datetime64[D]")

                    abs_diff = (
                        np.abs(sorted_par_dates - target_date)
                        .astype("timedelta64[D]")
                        .astype(int)
                    )

                    closest_index = np.argmin(abs_diff)

                    start_idx = max(0, closest_index - 5)
                    end_idx = min(len(sorted_par_files), closest_index + 6)

                    selected_files = sorted_par_files[start_idx:end_idx]
                    selected_dates = sorted_par_dates[start_idx:end_idx]

                    ds = None
                    par_data = None

                    try:
                        ds = xr.open_mfdataset(
                            selected_files,
                            concat_dim="time",
                            combine="nested",
                            chunks={},
                        )

                        ds["time"] = selected_dates.astype("datetime64[ns]")
                        ds = ds.sortby("time")

                        lat_idx = np.argmin(abs(ds.lat.values - lat_val))
                        lon_idx = np.argmin(abs(ds.lon.values - lon_val))

                        par_data = ds.isel(
                            lat=slice(
                                max(lat_idx - 2, 0),
                                min(lat_idx + 3, ds.sizes["lat"]),
                            ),
                            lon=slice(
                                max(lon_idx - 2, 0),
                                min(lon_idx + 3, ds.sizes["lon"]),
                            ),
                        )

                        par_value = bgc.get_nearest_valid_value(
                            par_data,
                            "PAR_mean",
                            lat_val,
                            lon_val,
                            dt,
                        )

                        xds["SAT_PAR"].values[n] = par_value

                    except Exception as e:
                        print(f"{name}: PAR lookup failed for profile {n}: {e}")
                        xds["SAT_PAR"].values[n] = np.nan

                    finally:
                        try:
                            if ds is not None:
                                ds.close()
                        except Exception:
                            pass

                        try:
                            del ds, par_data
                        except Exception:
                            pass

            # ------------------------------------------------------------
            # RRS/KD lookup only for missing/new profiles
            # ------------------------------------------------------------
            profile_lats = xds.LATITUDE.values.astype(float)
            profile_lons = xds.LONGITUDE.values.astype(float)
            profile_lons = ((profile_lons + 180) % 360) - 180
            profile_times = pd.to_datetime(xds.DATETIME.values)

            variables = ["RRS412", "RRS443", "RRS490", "RRS555", "RRS670"]

            lookup_idx = np.asarray(lookup_idx, dtype=int)
            profile_months = profile_times.to_period("M")

            if lookup_idx.size > 0:
                unique_months = np.sort(profile_months[lookup_idx].unique())
            else:
                unique_months = []

            dx = 0.036

            RRS_list = [
                {
                    var: (
                        float(xds[f"SAT_{var}"].values[i])
                        if np.isfinite(xds[f"SAT_{var}"].values[i])
                        else np.nan
                    )
                    for var in variables
                }
                for i in range(len(profile_lats))
            ]

            KD_list = list(xds["SAT_KD490"].values.astype(np.float32))

            LAT_MIN, LAT_MAX = -89.97917, 89.97917
            LON_MIN, LON_MAX = -179.97917, 179.97917

            for month_i, month in enumerate(unique_months, start=1):

                month_lookup_mask = profile_months[lookup_idx] == month
                month_idx = lookup_idx[month_lookup_mask]

                if month_idx.size == 0:
                    continue

                pbar.set_description(
                    f"File {file_i}/{len(flist_without_par)} | "
                    f"Lookup month {month_i}/{len(unique_months)}"
                )
                pbar.set_postfix_str(
                    f"{file_name} | {month} | {len(month_idx)} profiles"
                )

                lat_min = float(np.nanmin(profile_lats[month_idx]) - 2 * dx)
                lat_max = float(np.nanmax(profile_lats[month_idx]) + 2 * dx)
                lon_min = float(np.nanmin(profile_lons[month_idx]) - 2 * dx)
                lon_max = float(np.nanmax(profile_lons[month_idx]) + 2 * dx)

                lat_min = float(np.clip(lat_min, LAT_MIN, LAT_MAX))
                lat_max = float(np.clip(lat_max, LAT_MIN, LAT_MAX))
                lon_min = float(np.clip(lon_min, LON_MIN, LON_MAX))
                lon_max = float(np.clip(lon_max, LON_MIN, LON_MAX))

                time_min = profile_times[month_idx].min() - pd.Timedelta(days=5)
                time_max = profile_times[month_idx].max() + pd.Timedelta(days=5)

                regional_rrs = copernicusmarine.open_dataset(
                    dataset_id="cmems_obs-oc_glo_bgc-reflectance_my_l3-multi-4km_P1D",
                    start_datetime=str(time_min),
                    end_datetime=str(time_max),
                    minimum_latitude=lat_min,
                    maximum_latitude=lat_max,
                    minimum_longitude=lon_min,
                    maximum_longitude=lon_max,
                    variables=variables,
                ).load()

                regional_kd = copernicusmarine.open_dataset(
                    dataset_id="cmems_obs-oc_glo_bgc-transp_my_l3-multi-4km_P1D",
                    start_datetime=str(time_min),
                    end_datetime=str(time_max),
                    minimum_latitude=lat_min,
                    maximum_latitude=lat_max,
                    minimum_longitude=lon_min,
                    maximum_longitude=lon_max,
                    variables=["KD490"],
                ).load()

                rrs_lats = regional_rrs.latitude.values
                rrs_lons = regional_rrs.longitude.values

                for n in month_idx:

                    prof_lat = float(profile_lats[n])
                    prof_lon = float(profile_lons[n])
                    prof_time = np.datetime64(profile_times[n])

                    lat_idx = np.abs(rrs_lats - prof_lat).argmin()
                    lon_idx = np.abs(rrs_lons - prof_lon).argmin()

                    lat_slice = slice(
                        max(lat_idx - 2, 0),
                        min(lat_idx + 3, regional_rrs.sizes["latitude"]),
                    )
                    lon_slice = slice(
                        max(lon_idx - 2, 0),
                        min(lon_idx + 3, regional_rrs.sizes["longitude"]),
                    )
                    time_slice = slice(
                        prof_time - np.timedelta64(5, "D"),
                        prof_time + np.timedelta64(5, "D"),
                    )

                    sub_lat_vals = rrs_lats[lat_slice]
                    sub_lon_vals = rrs_lons[lon_slice]

                    cube_lats, cube_lons = np.meshgrid(
                        sub_lat_vals,
                        sub_lon_vals,
                        indexing="ij",
                    )

                    this_rrs = {}

                    for var in variables:
                        value = bgc.get_nearest_valid_value_cube(
                            regional_rrs,
                            var,
                            prof_lat,
                            prof_lon,
                            prof_time,
                            lat_slice,
                            lon_slice,
                            time_slice,
                            cube_lats,
                            cube_lons,
                        )

                        this_rrs[var] = value * np.pi if np.isfinite(value) else np.nan

                    this_kd = bgc.get_nearest_valid_value_cube(
                        regional_kd,
                        "KD490",
                        prof_lat,
                        prof_lon,
                        prof_time,
                        lat_slice,
                        lon_slice,
                        time_slice,
                        cube_lats,
                        cube_lons,
                    )

                    RRS_list[n] = this_rrs
                    KD_list[n] = this_kd

                regional_rrs.close()
                regional_kd.close()

                del regional_rrs, regional_kd
                gc.collect()

            RRS_df = pd.DataFrame(RRS_list)

            for var in variables:
                xds[f"SAT_{var}"] = xr.DataArray(
                    RRS_df[var].values,
                    dims="N_PROF",
                    coords={"N_PROF": xds.N_PROF},
                )

            xds["SAT_KD490"] = xr.DataArray(
                np.array(KD_list, dtype=np.float32),
                dims="N_PROF",
                coords={"N_PROF": xds.N_PROF},
            )

            # ------------------------------------------------------------
            # Irradiance profile reconstruction
            # ------------------------------------------------------------
            PAR_DATA = []
            ED380_DATA = []
            ED412_DATA = []
            ED490_DATA = []

            depth_output = np.linspace(0, 250, 51)

            for i in range(len(xds.N_PROF)):

                try:
                    LAT = xds.LATITUDE[i].values
                    LON = xds.LONGITUDE[i].values

                    timezone_str = tf.certain_timezone_at(lat=LAT, lng=LON)

                    if timezone_str is None:
                        raise ValueError(
                            f"Unable to find timezone for LAT={LAT}, LON={LON}"
                        )

                    timezone = pytz.timezone(timezone_str)

                    LC = pd.to_datetime(xds.DATETIME[i].values) + timezone.utcoffset(
                        pd.to_datetime(xds.DATETIME[i].values)
                    )

                    LC_Hour_Deci = LC.hour + (LC.minute / 60)

                    PAR = np.array(xds.SAT_PAR[i].values, dtype=np.float32)
                    MLD = np.array(xds.MLD[i].values, dtype=np.float32)
                    ZPD = 1 / np.array(xds.SAT_KD490[i].values, dtype=np.float32)

                    RHO412 = np.array(xds.SAT_RRS412[i].values, dtype=np.float32)
                    RHO443 = np.array(xds.SAT_RRS443[i].values, dtype=np.float32)
                    RHO490 = np.array(xds.SAT_RRS490[i].values, dtype=np.float32)
                    RHO555 = np.array(xds.SAT_RRS555[i].values, dtype=np.float32)
                    RHO670 = np.array(xds.SAT_RRS670[i].values, dtype=np.float32)

                    doy = np.array(LC.timetuple().tm_yday, dtype=np.float64)

                    sin_doy = np.sin(rad_date(doy))
                    cos_doy = np.cos(rad_date(doy))
                    sin_LC_hour = np.sin(rad_LC_hour(LC_Hour_Deci))
                    cos_LC_hour = np.cos(rad_LC_hour(LC_Hour_Deci))

                    temp = np.array(xds.TEMP[i, :250].values, dtype=np.float64)
                    sal = np.array(xds.PSAL[i, :250].values, dtype=np.float64)
                    pres = xds.PRES[:250].values

                    valid = (~np.isnan(pres)) & (~np.isnan(sal)) & (~np.isnan(temp))

                    pres2 = pres[valid]
                    temp2 = temp[valid]
                    sal2 = sal[valid]

                    pres3 = pres2[pres2 <= 250]
                    pres4 = pres2[pres2 <= 50]

                    if (len(pres3) >= 15) & (len(pres4) >= 5):

                        non_array_inputs = np.array(
                            [
                                PAR,
                                MLD,
                                ZPD,
                                RHO412,
                                RHO443,
                                RHO490,
                                RHO555,
                                RHO670,
                                sin_doy,
                                cos_doy,
                                sin_LC_hour,
                                cos_LC_hour,
                            ],
                            dtype=np.float64,
                        )

                        if any(np.isnan(x).any() for x in non_array_inputs):

                            PAR_ADJUSTED = np.full(51, np.nan)
                            ED380_ADJUSTED = np.full(51, np.nan)
                            ED412_ADJUSTED = np.full(51, np.nan)
                            ED490_ADJUSTED = np.full(51, np.nan)

                        else:

                            PAR_ADJUSTED = PAR_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED380_ADJUSTED = ED380_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED412_ADJUSTED = ED412_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                            ED490_ADJUSTED = ED490_ADJUSTED_profiles(
                                PAR=PAR,
                                MLD=MLD,
                                ZPD=ZPD,
                                RHO_WN_412=RHO412,
                                RHO_WN_443=RHO443,
                                RHO_WN_490=RHO490,
                                RHO_WN_555=RHO555,
                                RHO_WN_670=RHO670,
                                sin_doy=sin_doy,
                                cos_doy=cos_doy,
                                sin_LC_hour=sin_LC_hour,
                                cos_LC_hour=cos_LC_hour,
                                temp=temp2,
                                sal=sal2,
                                depths_argo=pres2,
                                depth_output=depth_output,
                            )[0]

                    else:
                        PAR_ADJUSTED = np.full(51, np.nan)
                        ED380_ADJUSTED = np.full(51, np.nan)
                        ED412_ADJUSTED = np.full(51, np.nan)
                        ED490_ADJUSTED = np.full(51, np.nan)

                    PAR_DATA.append(PAR_ADJUSTED)
                    ED380_DATA.append(ED380_ADJUSTED)
                    ED412_DATA.append(ED412_ADJUSTED)
                    ED490_DATA.append(ED490_ADJUSTED)

                except Exception as e:
                    print(f"Error processing profile {i}: {e}")

                    PAR_DATA.append(np.full(51, np.nan))
                    ED380_DATA.append(np.full(51, np.nan))
                    ED412_DATA.append(np.full(51, np.nan))
                    ED490_DATA.append(np.full(51, np.nan))

            PAR_DATA = pd.DataFrame(np.array(PAR_DATA).T)
            ED380_DATA = pd.DataFrame(np.array(ED380_DATA).T)
            ED412_DATA = pd.DataFrame(np.array(ED412_DATA).T)
            ED490_DATA = pd.DataFrame(np.array(ED490_DATA).T)

            depths = np.linspace(0, 250, 51)

            PAR_DATA.index = depths
            ED380_DATA.index = depths
            ED412_DATA.index = depths
            ED490_DATA.index = depths

            PAR_DATA = PAR_DATA.reindex(xds.PRES.values).interpolate()
            ED380_DATA = ED380_DATA.reindex(xds.PRES.values).interpolate()
            ED412_DATA = ED412_DATA.reindex(xds.PRES.values).interpolate()
            ED490_DATA = ED490_DATA.reindex(xds.PRES.values).interpolate()

            xds["DOWNWELLING_PAR"] = xr.DataArray(
                np.array(PAR_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds = bgc.clean_range_filter(xds, "DOWNWELLING_PAR", 0.001, 4672)

            xds["DOWN_IRRADIANCE380"] = xr.DataArray(
                np.array(ED380_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds["DOWN_IRRADIANCE412"] = xr.DataArray(
                np.array(ED412_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds["DOWN_IRRADIANCE490"] = xr.DataArray(
                np.array(ED490_DATA.T),
                dims=("N_PROF", "PRES"),
                coords={"N_PROF": xds.N_PROF, "PRES": xds.PRES},
            )

            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE380", 1e-5, 1.7)
            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE412", 1e-5, 2.9)
            xds = bgc.clean_range_filter(xds, "DOWN_IRRADIANCE490", 1e-5, 3.4)

            # ------------------------------------------------------------
            # Euphotic depth
            # ------------------------------------------------------------
            par = xds["DOWNWELLING_PAR"].sel(PRES=slice(0, 199))
            depth = xds["PRES"].sel(PRES=slice(0, 199))

            slope, r2, n, zeu = xr.apply_ufunc(
                bgc.euphotic_depth,
                par,
                depth,
                input_core_dims=[["PRES"], ["PRES"]],
                output_core_dims=[[], [], [], []],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float, float, float],
                kwargs={
                    "initial_fit_depth": 100,
                    "max_fit_depth": 186,
                    "step": 20,
                },
            )

            xds = xds.assign(
                KD_PAR=slope,
                KD_PAR_R2=r2,
                ZEU=zeu,
            )

            # ------------------------------------------------------------
            # Chlorophyll fluorescence quenching
            # ------------------------------------------------------------
            xds["SOLAR_ELEVATION"] = bgc.calculate_solar_elevation(xds)

            ds = xds.sel(PRES=slice(0, 199))

            result = xr.apply_ufunc(
                bgc.apply_xing_quenching_correction,
                ds["CHLA_FLUORESCENCE"],
                ds["BBP_PHY"],
                ds["DOWNWELLING_PAR"],
                ds["MLD"],
                ds["SOLAR_ELEVATION"],
                ds["PRES"],
                input_core_dims=[
                    ["PRES"],
                    ["PRES"],
                    ["PRES"],
                    [],
                    [],
                    ["PRES"],
                ],
                output_core_dims=[["PRES"], ["PRES"], [], []],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float, float, float, bool],
            )

            xds = xds.assign(
                CHLA=result[0],
                NPQ=result[1],
                QUENCHING_DEPTH=result[2],
                QUENCHING_FLAG=result[3],
            )

            xds["CHLA"] = xds.CHLA / 2
            xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

            # ------------------------------------------------------------
            # NPP
            # ------------------------------------------------------------
            xds["PAR_DAILY"] = bgc.compute_daily_par_clear_sky(xds, dt_seconds=60, par_top_depth=10)
            ds = xds.sel(PRES=slice(0, 199))

            result = xr.apply_ufunc(
                bgc.cbpm_argo,
                ds["CHLA"],
                ds["CPHYTO"],
                ds["PAR_DAILY"],
                ds["DATETIME"].dt.dayofyear,
                ds["LATITUDE"],
                input_core_dims=[["PRES"], ["PRES"], [], [], []],
                output_core_dims=[["PRES"], ["PRES"], ["PRES"], ["PRES"]],
                vectorize=True,
                dask="parallelized",
                output_dtypes=[float] * 4,
            )

            xds = xds.assign(
                WESTBERRY_CBPM=result[0],
                WESTBERRY_MU=result[1],
                WESTBERRY_NUT_TEMP_FUNC=result[2],
                WESTBERRY_IG_FUNC=result[3],
            )

            xds["WESTBERRY_CBPM"] = xds["WESTBERRY_CBPM"].where(
                xds["WESTBERRY_CBPM"] >= 0,
                np.nan,
            )

            var_list = ["CHLA", "CPHYTO", "WESTBERRY_CBPM"]

            int_all = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(xds.PRES <= 199, drop=False),
                    var_list,
                )
                .rename({v: f"{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)

            int_zeu = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(zeu_mask, drop=False),
                    var_list,
                )
                .rename({v: f"ZEU_{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)

            int_mld = (
                bgc.integrate_along_pres_multiple_vars(
                    xds.where(mld_mask, drop=False),
                    var_list,
                )
                .rename({v: f"MLD_{v}_INT" for v in var_list})
                .where(lambda ds: ds != 0, np.nan)
            )

            int_all = xr.merge([int_all, int_zeu, int_mld], join="outer")

            mean_vars = [
                "WESTBERRY_CBPM",
                "WESTBERRY_IG_FUNC",
                "WESTBERRY_NUT_TEMP_FUNC",
                "CHLA",
                "CPHYTO",
                "TEMP",
                "DOWNWELLING_PAR",
            ]

            data_eu = xds[mean_vars].where(
                xds.PRES <= xds.ZEU,
                drop=False,
            ).mean(dim="PRES")

            int_all["ZEU_WESTBERRY_CBPM"] = data_eu["WESTBERRY_CBPM"]
            int_all["ZEU_WESTBERRY_IG_FUNC"] = data_eu["WESTBERRY_IG_FUNC"]
            int_all["ZEU_WESTBERRY_NUT_TEMP_FUNC"] = data_eu["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["ZEU_CHLA"] = data_eu["CHLA"]
            int_all["ZEU_CPHYTO"] = data_eu["CPHYTO"]
            int_all["ZEU_TEMP"] = data_eu["TEMP"]
            int_all["ZEU_DOWNWELLING_PAR"] = data_eu["DOWNWELLING_PAR"]

            data_mld = xds[mean_vars].where(
                xds.PRES <= xds.MLD,
                drop=False,
            ).mean(dim="PRES")

            int_all["MLD_WESTBERRY_CBPM"] = data_mld["WESTBERRY_CBPM"]
            int_all["MLD_WESTBERRY_IG_FUNC"] = data_mld["WESTBERRY_IG_FUNC"]
            int_all["MLD_WESTBERRY_NUT_TEMP_FUNC"] = data_mld["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["MLD_CHLA"] = data_mld["CHLA"]
            int_all["MLD_CPHYTO"] = data_mld["CPHYTO"]
            int_all["MLD_TEMP"] = data_mld["TEMP"]
            int_all["MLD_DOWNWELLING_PAR"] = data_mld["DOWNWELLING_PAR"]

            od = abs(1 / xds.KD_PAR)

            data_od = xds[mean_vars].where(
                xds.PRES <= od,
                drop=False,
            ).mean(dim="PRES")

            int_all["OD1_WESTBERRY_CBPM"] = data_od["WESTBERRY_CBPM"]
            int_all["OD1_WESTBERRY_IG_FUNC"] = data_od["WESTBERRY_IG_FUNC"]
            int_all["OD1_WESTBERRY_NUT_TEMP_FUNC"] = data_od["WESTBERRY_NUT_TEMP_FUNC"]
            int_all["OD1_CHLA"] = data_od["CHLA"]
            int_all["OD1_CPHYTO"] = data_od["CPHYTO"]
            int_all["OD1_TEMP"] = data_od["TEMP"]
            int_all["OD1_DOWNWELLING_PAR"] = data_od["DOWNWELLING_PAR"]

            int_all["DATETIME"] = xds.DATETIME
            int_all["LATITUDE"] = xds.LATITUDE
            int_all["LONGITUDE"] = xds.LONGITUDE
            int_all["ZEU"] = xds.ZEU
            int_all["MLD"] = xds.MLD
            int_all["SOLAR_ELEVATION"] = xds.SOLAR_ELEVATION
            int_all["QUENCHING_FLAG"] = xds.QUENCHING_FLAG

            if int_all.N_PROF.shape[0] == 0:
                skipped_files.append(name)
                bgc.remove_from_pending(pending_file, file)
                pbar.update(1)
                continue

            xds["BIOME"] = biomes.interp(
                lat=xds["LATITUDE"],
                lon=xds["LONGITUDE"],
                method="nearest",
            )

            int_all["BIOME"] = biomes.interp(
                lat=int_all["LATITUDE"],
                lon=int_all["LONGITUDE"],
                method="nearest",
            )

            drop_from_xds = [v for v in ["lat", "lon"] if v in xds]
            drop_from_int = [v for v in ["lat", "lon"] if v in int_all]

            if drop_from_xds:
                xds = xds.drop_vars(drop_from_xds)

            if drop_from_int:
                int_all = int_all.drop_vars(drop_from_int)

            xds.to_netcdf(
                tmp_profile_file,
                format="NETCDF4",
                encoding=bgc.make_encoding(xds, complevel=1),
            )

            int_all.to_netcdf(
                tmp_npp_file,
                format="NETCDF4",
                encoding=bgc.make_encoding(int_all, complevel=1),
            )

            tmp_profile_file.replace(final_profile_file)
            tmp_npp_file.replace(final_npp_file)

            bgc.remove_from_pending(pending_file, file)

            pbar.update(1)

            try:
                xds.close()
                int_all.close()
            except Exception:
                pass

            del xds, int_all
            gc.collect()

        except Exception as e:
            print(f"❌ Error processing {name}: {e}")
            skipped_files.append(name)
            pbar.update(1)

            try:
                xds.close()
            except Exception:
                pass

            try:
                int_all.close()
            except Exception:
                pass

            gc.collect()
            continue

File 1/335:   0%|                                                             | 0/335 [00:00<?, ?it/s, 7902134_Sprof.nc]

7902134_Sprof: 50 profiles total | 6 missing SAT lookups | last profile 2026-04-20 | 6 eligible for new lookup | skip inactive = True


File 2/335:   0%|▏                                                  | 1/335 [00:44<4:05:10, 44.04s/it, 7902135_Sprof.nc]

7902135_Sprof: 16 profiles total | 0 missing SAT lookups | last profile 2025-06-05 | 0 eligible for new lookup | skip inactive = True


File 3/335:   1%|▎                                                  | 2/335 [00:48<1:55:03, 20.73s/it, 7902136_Sprof.nc]

...

File 334/335:  99%|██████████████████████████████████████████████▋| 333/335 [1:40:31<01:43, 51.61s/it, 4902681_Sprof.nc]

4902681_Sprof: 338 profiles total | 75 missing SAT lookups | last profile 2026-04-28 | 75 eligible for new lookup | skip inactive = True


File 335/335: 100%|██████████████████████████████████████████████▊| 334/335 [1:43:58<01:38, 98.32s/it, 4902683_Sprof.nc]

4902683_Sprof: 349 profiles total | 85 missing SAT lookups | last profile 2026-04-26 | 85 eligible for new lookup | skip inactive = True


File 335/335 | Lookup month 5/5: 100%|████| 335/335 [1:47:33<00:00, 19.26s/it, 4902683_Sprof.nc | 2026-04 | 16 profiles]


### Deriving F/Chl Conversion Factors

#### Float Specific - With Irradiance Sensors

In [7]:
fname = '../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/*.nc'
flist = sorted(glob.glob(fname))

outdir = Path('../DATA/Output Final/Profiles_SOCA/KD490/XING/DAILY_WITH_IRRADIANCE')
outdir_npp = Path('../DATA/Output Final/NPP_SOCA/KD490/XING/DAILY_WITH_IRRADIANCE')

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist):

    name = os.path.basename(file).split('.')[0]

    out_file = outdir / f"{name}.nc"
    out_file_npp = outdir_npp / f"{name}_NPP.nc"

    # Always rebuild outputs so delayed-mode updates are included
    #if out_file.exists() or out_file_npp.exists():
    #    print(f"{name}: rebuilding and replacing existing outputs.")

    xds = xr.open_dataset(file)
    xds['CHLA'] = xds.CHLA * 2

    result = xr.apply_ufunc(bgc.compute_fluorescence_to_chla_factor,
                             xds['DOWN_IRRADIANCE490'],
                             xds['CHLA'],
                             xds['SOLAR_ELEVATION'],
                             input_core_dims=[["PRES"], ["PRES"], []],
                             output_core_dims=[[], [], [], [], ["PRES"], ["PRES"]],
                             vectorize=True,
                             dask="parallelized",
                             output_dtypes=[float, float, float, float, float, float],)

    xds['CHLA_CONVERSION_FACTOR'] = result[0]
    xds['CHLA_CONVERSION_FACTOR_RSQ'] = result[3]
    xds['CHLA_CF_CN'] = result[4]
    xds['CHLA_CF_AN'] = result[5]

    # ---------------------------------------------------------
    # Replace missing conversion factors with float-wide median
    # ---------------------------------------------------------
    cf = xds["CHLA_CONVERSION_FACTOR"].values.astype(float)
    
    if np.all(np.isnan(cf)):
        name = os.path.basename(file)
        print(f"{name}: no valid conversion factors.")
        continue  # Skip float entirely if no valid conversion factors
    
    # Compute robust central tendency
    median = np.nanmedian(cf)
    mad = median_abs_deviation(cf, nan_policy="omit")
    
    # Outlier filtering using modified z-score
    if mad == 0 or np.isnan(mad):
        mask = np.isfinite(cf)
    else:
        z = np.abs((cf - median) / (1.4826 * mad))
        mask = np.isfinite(cf) & (z < 3)
    
    cf_filtered = np.where(mask, cf, np.nan)
    
    xds["CHLA_CF_FILTERED"] = (("N_PROF",), cf_filtered)
    
    mask_da = xr.DataArray(mask, dims=("N_PROF",), coords={"N_PROF": xds["N_PROF"]})
    
    xds["CHLA_CF_FILTERED_RSQ"] = xds["CHLA_CONVERSION_FACTOR_RSQ"].where(mask_da)
    
    # Replace missing/outlier values with float-wide median of filtered CF
    median_cf = np.nanmedian(cf_filtered)
    
    if np.isnan(median_cf):
        name = os.path.basename(file)
        print(f"{name}: no valid conversion factors after filtering.")
        continue  # Safety: all values were filtered out
    
    xds["CHLA_CF_FINAL"] = xds["CHLA_CF_FILTERED"].fillna(median_cf)

    # Apply conversion factor to CHLA
    xds["CHLA"] = xds["CHLA"] * xds["CHLA_CF_FINAL"]

    # Apply range filter to remove spurious CHLA
    xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)
        
    # Apply NPP algorithms
    ds = xds.sel(PRES=slice(0, 199))
    result = xr.apply_ufunc(bgc.cbpm_argo,
                            ds['CHLA'],
                            ds['CPHYTO'],
                            ds['PAR_DAILY'],
                            ds['DATETIME'].dt.dayofyear,
                            ds['LATITUDE'],
                            input_core_dims=[['PRES'], ['PRES'], [], [], []],
                            output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                            vectorize=True,
                            dask='parallelized',
                            output_dtypes=[float] * 4)

    xds = xds.assign(WESTBERRY_CBPM=result[0],
                     WESTBERRY_MU=result[1],
                     WESTBERRY_NUT_TEMP_FUNC=result[2],
                     WESTBERRY_IG_FUNC=result[3])

    xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] >= 0, np.nan)

    var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM']
        
    # Integrate to 200 m
    int_all = bgc.integrate_along_pres_multiple_vars(
        xds.where(xds.PRES <= 199, drop=False), var_list
    ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to Zeu
    zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
    int_zeu = bgc.integrate_along_pres_multiple_vars(
        xds.where(zeu_mask, drop=False), var_list
    ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to MLD
    mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
    int_mld = bgc.integrate_along_pres_multiple_vars(
        xds.where(mld_mask, drop=False), var_list
    ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Merge safely
    int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')

    data_eu = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
    int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
    int_all['ZEU_WESTBERRY_IG_FUNC'] = data_eu['WESTBERRY_IG_FUNC']
    int_all['ZEU_WESTBERRY_NUT_TEMP_FUNC'] = data_eu['WESTBERRY_NUT_TEMP_FUNC']
    int_all['ZEU_CHLA'] = data_eu['CHLA']
    int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
    int_all['ZEU_TEMP'] = data_eu['TEMP']
    int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']

    data_mld = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
    int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
    int_all['MLD_WESTBERRY_IG_FUNC'] = data_mld['WESTBERRY_IG_FUNC']
    int_all['MLD_WESTBERRY_NUT_TEMP_FUNC'] = data_mld['WESTBERRY_NUT_TEMP_FUNC']
    int_all['MLD_CHLA'] = data_mld['CHLA']
    int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
    int_all['MLD_TEMP'] = data_mld['TEMP']
    int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']

    od = abs(1 / xds.KD_PAR)
    data_od = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
    int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
    int_all['OD1_WESTBERRY_IG_FUNC'] = data_od['WESTBERRY_IG_FUNC']
    int_all['OD1_WESTBERRY_NUT_TEMP_FUNC'] = data_od['WESTBERRY_NUT_TEMP_FUNC']
    int_all['OD1_CHLA'] = data_od['CHLA']
    int_all['OD1_CPHYTO'] = data_od['CPHYTO']
    int_all['OD1_TEMP'] = data_od['TEMP']
    int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
    
    int_all['DATETIME'] = xds.DATETIME
    int_all['LATITUDE'] = xds.LATITUDE
    int_all['LONGITUDE'] = xds.LONGITUDE
    int_all['ZEU'] = xds.ZEU
    int_all['MLD'] = xds.MLD
    int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
    int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG

    if int_all.N_PROF.shape[0] == 0:
        continue

    xds['BIOME'] = biomes.interp(
                lat=xds['LATITUDE'],
                lon=xds['LONGITUDE'],
                method='nearest')

    int_all['BIOME'] = biomes.interp(
        lat=int_all['LATITUDE'],
        lon=int_all['LONGITUDE'],
        method='nearest')

    drop_from_xds = [v for v in ['lat', 'lon'] if v in xds]
    drop_from_int = [v for v in ['lat', 'lon'] if v in int_all]

    if drop_from_xds:
        xds = xds.drop_vars(drop_from_xds)

    if drop_from_int:
        int_all = int_all.drop_vars(drop_from_int)
        
    xds.to_netcdf(
        out_file,
        format="NETCDF4",
        encoding=bgc.make_encoding(xds, complevel=1),
        mode="w",
    )

    int_all.to_netcdf(
        out_file_npp,
        format="NETCDF4",
        encoding=bgc.make_encoding(int_all, complevel=1),
        mode="w",
    )

  2%|█▊                                                                                 | 8/362 [00:02<01:39,  3.55it/s]

1902651_Sprof.nc: no valid conversion factors.


  5%|████                                                                              | 18/362 [00:07<02:02,  2.80it/s]

2902823_Sprof.nc: no valid conversion factors.


  6%|████▊                                                                             | 21/362 [00:07<01:25,  3.97it/s]

...

 96%|██████████████████████████████████████████████████████████████████████████████   | 349/362 [02:12<00:02,  6.20it/s]

7901106_Sprof.nc: no valid conversion factors.


 99%|████████████████████████████████████████████████████████████████████████████████▎| 359/362 [02:13<00:00,  9.52it/s]

7902198_Sprof.nc: no valid conversion factors.


100%|█████████████████████████████████████████████████████████████████████████████████| 362/362 [02:13<00:00,  2.71it/s]


#### Float Specific - No Irradiance Sensors

In [8]:
fname = '../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITHOUT_IRRADIANCE/*.nc'
flist = sorted(glob.glob(fname))

outdir = Path('../DATA/Output Final/Profiles_SOCA/KD490/XING/DAILY_WITHOUT_IRRADIANCE')
outdir_npp = Path('../DATA/Output Final/NPP_SOCA/KD490/XING/DAILY_WITHOUT_IRRADIANCE')

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist):

    name = os.path.basename(file).split('.')[0]

    out_file = outdir / f"{name}.nc"
    out_file_npp = outdir_npp / f"{name}_NPP.nc"

    # Always rebuild outputs so delayed-mode updates are included
    #if out_file.exists() or out_file_npp.exists():
    #    print(f"{name}: rebuilding and replacing existing outputs.")

    xds = xr.open_dataset(file)
    xds['CHLA'] = xds.CHLA * 2

    result = xr.apply_ufunc(bgc.compute_fluorescence_to_chla_factor,
                             xds['DOWN_IRRADIANCE490'],
                             xds['CHLA'],
                             xds['SOLAR_ELEVATION'],
                             input_core_dims=[["PRES"], ["PRES"], []],
                             output_core_dims=[[], [], [], [], ["PRES"], ["PRES"]],
                             vectorize=True,
                             dask="parallelized",
                             output_dtypes=[float, float, float, float, float, float],)

    xds['CHLA_CONVERSION_FACTOR'] = result[0]
    xds['CHLA_CONVERSION_FACTOR_RSQ'] = result[3]
    xds['CHLA_CF_CN'] = result[4]
    xds['CHLA_CF_AN'] = result[5]

    # ---------------------------------------------------------
    # Replace missing conversion factors with float-wide median
    # ---------------------------------------------------------
    cf = xds["CHLA_CONVERSION_FACTOR"].values.astype(float)
    
    if np.all(np.isnan(cf)):
        name = os.path.basename(file)
        print(f"{name}: no valid conversion factors.")
        continue  # Skip float entirely if no valid conversion factors
    
    # Compute robust central tendency
    median = np.nanmedian(cf)
    mad = median_abs_deviation(cf, nan_policy="omit")
    
    # Outlier filtering using modified z-score
    if mad == 0 or np.isnan(mad):
        mask = np.isfinite(cf)
    else:
        z = np.abs((cf - median) / (1.4826 * mad))
        mask = np.isfinite(cf) & (z < 3)
    
    cf_filtered = np.where(mask, cf, np.nan)
    
    xds["CHLA_CF_FILTERED"] = (("N_PROF",), cf_filtered)
    
    mask_da = xr.DataArray(mask, dims=("N_PROF",), coords={"N_PROF": xds["N_PROF"]})
    
    xds["CHLA_CF_FILTERED_RSQ"] = xds["CHLA_CONVERSION_FACTOR_RSQ"].where(mask_da)
    
    # Replace missing/outlier values with float-wide median of filtered CF
    median_cf = np.nanmedian(cf_filtered)
    
    if np.isnan(median_cf):
        name = os.path.basename(file)
        print(f"{name}: no valid conversion factors after filtering.")
        continue  # Safety: all values were filtered out
    
    xds["CHLA_CF_FINAL"] = xds["CHLA_CF_FILTERED"].fillna(median_cf)

    # Apply conversion factor to CHLA
    xds["CHLA"] = xds["CHLA"] * xds["CHLA_CF_FINAL"]

    # Apply range filter to remove spurious CHLA
    xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

    # Apply NPP algorithms
    ds = xds.sel(PRES=slice(0, 199))
    result = xr.apply_ufunc(bgc.cbpm_argo,
                            ds['CHLA'],
                            ds['CPHYTO'],
                            ds['PAR_DAILY'],
                            ds['DATETIME'].dt.dayofyear,
                            ds['LATITUDE'],
                            input_core_dims=[['PRES'], ['PRES'], [], [], []],
                            output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                            vectorize=True,
                            dask='parallelized',
                            output_dtypes=[float] * 4)

    xds = xds.assign(WESTBERRY_CBPM=result[0],
                     WESTBERRY_MU=result[1],
                     WESTBERRY_NUT_TEMP_FUNC=result[2],
                     WESTBERRY_IG_FUNC=result[3])

    xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] >= 0, np.nan)

    var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM']
        
    # Integrate to 200 m
    int_all = bgc.integrate_along_pres_multiple_vars(
        xds.where(xds.PRES <= 199, drop=False), var_list
    ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to Zeu
    zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
    int_zeu = bgc.integrate_along_pres_multiple_vars(
        xds.where(zeu_mask, drop=False), var_list
    ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to MLD
    mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
    int_mld = bgc.integrate_along_pres_multiple_vars(
        xds.where(mld_mask, drop=False), var_list
    ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Merge safely
    int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')

    data_eu = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
    int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
    int_all['ZEU_WESTBERRY_IG_FUNC'] = data_eu['WESTBERRY_IG_FUNC']
    int_all['ZEU_WESTBERRY_NUT_TEMP_FUNC'] = data_eu['WESTBERRY_NUT_TEMP_FUNC']
    int_all['ZEU_CHLA'] = data_eu['CHLA']
    int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
    int_all['ZEU_TEMP'] = data_eu['TEMP']
    int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']

    data_mld = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
    int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
    int_all['MLD_WESTBERRY_IG_FUNC'] = data_mld['WESTBERRY_IG_FUNC']
    int_all['MLD_WESTBERRY_NUT_TEMP_FUNC'] = data_mld['WESTBERRY_NUT_TEMP_FUNC']
    int_all['MLD_CHLA'] = data_mld['CHLA']
    int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
    int_all['MLD_TEMP'] = data_mld['TEMP']
    int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']

    od = abs(1 / xds.KD_PAR)
    data_od = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
    int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
    int_all['OD1_WESTBERRY_IG_FUNC'] = data_od['WESTBERRY_IG_FUNC']
    int_all['OD1_WESTBERRY_NUT_TEMP_FUNC'] = data_od['WESTBERRY_NUT_TEMP_FUNC']
    int_all['OD1_CHLA'] = data_od['CHLA']
    int_all['OD1_CPHYTO'] = data_od['CPHYTO']
    int_all['OD1_TEMP'] = data_od['TEMP']
    int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
    
    int_all['DATETIME'] = xds.DATETIME
    int_all['LATITUDE'] = xds.LATITUDE
    int_all['LONGITUDE'] = xds.LONGITUDE
    int_all['ZEU'] = xds.ZEU
    int_all['MLD'] = xds.MLD
    int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
    int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG

    if int_all.N_PROF.shape[0] == 0:
        continue

    xds['BIOME'] = biomes.interp(
                lat=xds['LATITUDE'],
                lon=xds['LONGITUDE'],
                method='nearest')

    int_all['BIOME'] = biomes.interp(
        lat=int_all['LATITUDE'],
        lon=int_all['LONGITUDE'],
        method='nearest')

    drop_from_xds = [v for v in ['lat', 'lon'] if v in xds]
    drop_from_int = [v for v in ['lat', 'lon'] if v in int_all]

    if drop_from_xds:
        xds = xds.drop_vars(drop_from_xds)

    if drop_from_int:
        int_all = int_all.drop_vars(drop_from_int)

    xds.to_netcdf(
        out_file,
        format="NETCDF4",
        encoding=bgc.make_encoding(xds, complevel=1),
        mode="w",
    )

    int_all.to_netcdf(
        out_file_npp,
        format="NETCDF4",
        encoding=bgc.make_encoding(int_all, complevel=1),
        mode="w",
    )

  0%|▎                                                                                  | 3/926 [00:00<01:04, 14.31it/s]

1901338_Sprof.nc: no valid conversion factors.
1901339_Sprof.nc: no valid conversion factors.


  1%|▍                                                                                  | 5/926 [00:00<01:29, 10.25it/s]

1901614_Sprof.nc: no valid conversion factors.


  1%|▋                                                                                  | 7/926 [00:00<01:42,  8.97it/s]

...

7902250_Sprof.nc: no valid conversion factors.


 99%|███████████████████████████████████████████████████████████████████████████████▊ | 913/926 [02:42<00:01, 12.73it/s]

7902268_Sprof.nc: no valid conversion factors.
7902269_Sprof.nc: no valid conversion factors.


100%|█████████████████████████████████████████████████████████████████████████████████| 926/926 [02:44<00:00,  5.64it/s]

7902382_Sprof.nc: no valid conversion factors.


#### Generate Climatology

In [9]:
fname = '../DATA/Output Final/Profiles_SOCA/KD490/XING/*/*.nc'
flist = sorted(glob.glob(fname))

res = []

for file in tqdm(flist):

    xds = xr.open_dataset(file)
    
    df = xds[['CHLA_CONVERSION_FACTOR', 'CHLA_CONVERSION_FACTOR_RSQ', 
              'CHLA_CF_FILTERED', 'CHLA_CF_FINAL',
              'BIOME', 'LATITUDE', 'LONGITUDE', 'DATETIME']].to_dataframe().reset_index()

    df['FLOAT'] = int(file.split('/')[-1].split('.')[0].split('_')[0])
    df = df.set_index(['FLOAT', 'N_PROF'])
    res.append(df)

# Combine results
res = pd.concat(res)

res = res.copy()

res["MONTH"] = res["DATETIME"].dt.month

# keep original for diagnostics
res["CHLA_CF_FILTERED_ORIG"] = res["CHLA_CF_FILTERED"]

# invert safely
valid = np.isfinite(res["CHLA_CF_FILTERED"]) & (res["CHLA_CF_FILTERED"] > 0)

res["CHLA_CF_FILTERED_INV"] = np.nan
res.loc[valid, "CHLA_CF_FILTERED_INV"] = 1 / res.loc[valid, "CHLA_CF_FILTERED"]

summary = (
    res
    .groupby(["BIOME", "MONTH"])
    .agg(
        n_profiles=("CHLA_CF_FILTERED_ORIG", "size"),
        n_valid_orig=("CHLA_CF_FILTERED_ORIG", lambda x: np.isfinite(x).sum()),
        n_valid_inv=("CHLA_CF_FILTERED_INV", lambda x: np.isfinite(x).sum()),
        mean_inv=("CHLA_CF_FILTERED_INV", "mean"),
    )
)

summary["XING_FACTOR"] = 1 / summary["mean_inv"]

group_xing = summary.loc[summary["n_valid_inv"] >= 3, "XING_FACTOR"]
group_xing.unstack().to_csv(
    "../DATA/BIOME_MONTHLY_CHLA_CONVERSION_FACTOR_XING.csv"
)

100%|███████████████████████████████████████████████████████████████████████████████| 1105/1105 [00:11<00:00, 95.65it/s]


#### Apply Climatology - With Irradiance Sensors

In [10]:
fname = '../DATA/BIOME_MONTHLY_CHLA_CONVERSION_FACTOR_XING.csv'
MONTHLY_CLIM = pd.read_csv(fname, index_col="BIOME")
# Ensure integer biome/month labels
MONTHLY_CLIM.index = MONTHLY_CLIM.index.astype(int)
MONTHLY_CLIM.columns = MONTHLY_CLIM.columns.astype(int)


In [11]:
fname = '../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/*.nc'
flist = sorted(glob.glob(fname))

outdir = Path('../DATA/Output Final/Profiles_SOCA/KD490/XING_CLIM/DAILY_WITH_IRRADIANCE')
outdir_npp = Path('../DATA/Output Final/NPP_SOCA/KD490/XING_CLIM/DAILY_WITH_IRRADIANCE')

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist):
    # Get the output file path
    out_file = os.path.join(outdir, os.path.basename(file))  # You can modify this if you want a different naming convention
    
    # Check if the file has already been processed
    if os.path.exists(out_file):
        #print(f"Skipping {file}, already processed.")
        continue  # Skip this file and move to the next one

    name = os.path.basename(file).split('.')[0]

    xds = xr.open_dataset(file)

    xds["CHLA"] = xds["CHLA"].transpose("N_PROF", "PRES")
    
    # --------------------------------------------------
    # 1. Extract biome and month per profile
    # --------------------------------------------------
    biome = np.nan_to_num(xds["BIOME"].values, nan=-1).astype(int)
    month = xds["DATETIME"].dt.month.values.astype(int)
    
    # --------------------------------------------------
    # 2. Vectorised lookup of monthly biome factor
    # --------------------------------------------------
    lookup = MONTHLY_CLIM.stack().rename("CHLA_SLOPE_FACTOR").reset_index()
    lookup = lookup.rename(columns={"level_1": "MONTH"})
    
    factor_df = pd.DataFrame({
        "N_PROF": np.arange(xds.sizes["N_PROF"]),
        "BIOME": biome,
        "MONTH": month,})
    
    factor_df = factor_df.merge(
        lookup,
        on=["BIOME", "MONTH"],
        how="left",)
    
    slope = factor_df["CHLA_SLOPE_FACTOR"].to_numpy(dtype="float32")

    # --------------------------------------------------
    # Skip file if no valid CHLA slope factor was found
    # --------------------------------------------------
    if np.all(~np.isfinite(slope)):
        #print(f"Skipping {Path(file).name}: no valid CHLA_SLOPE_FACTOR found")
        xds.close()
        continue
    
    # --------------------------------------------------
    # 3. Assign factor to dataset
    # --------------------------------------------------
    xds = xds.assign(CHLA_SLOPE_FACTOR=("N_PROF", slope))
    
    # --------------------------------------------------
    # 4. Remove Roesler factor, apply Xing biome/month factor
    # --------------------------------------------------
    xds["CHLA"] = xds["CHLA"] * 2.0
    xds["CHLA"] = xds["CHLA"] * xds["CHLA_SLOPE_FACTOR"]
    
    # --------------------------------------------------
    # 5. Clean corrected chlorophyll
    # --------------------------------------------------
    xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

    # Apply NPP algorithms
    ds = xds.sel(PRES=slice(0, 199))
    result = xr.apply_ufunc(bgc.cbpm_argo,
                            ds['CHLA'],
                            ds['CPHYTO'],
                            ds['PAR_DAILY'],
                            ds['DATETIME'].dt.dayofyear,
                            ds['LATITUDE'],
                            input_core_dims=[['PRES'], ['PRES'], [], [], []],
                            output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                            vectorize=True,
                            dask='parallelized',
                            output_dtypes=[float] * 4)

    xds = xds.assign(WESTBERRY_CBPM=result[0],
                     WESTBERRY_MU=result[1],
                     WESTBERRY_NUT_TEMP_FUNC=result[2],
                     WESTBERRY_IG_FUNC=result[3])

    xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] >= 0, np.nan)

    var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM']
        
    # Integrate to 200 m
    int_all = bgc.integrate_along_pres_multiple_vars(
        xds.where(xds.PRES <= 199, drop=False), var_list
    ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to Zeu
    zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
    int_zeu = bgc.integrate_along_pres_multiple_vars(
        xds.where(zeu_mask, drop=False), var_list
    ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to MLD
    mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
    int_mld = bgc.integrate_along_pres_multiple_vars(
        xds.where(mld_mask, drop=False), var_list
    ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Merge safely
    int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')

    data_eu = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
    int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
    int_all['ZEU_WESTBERRY_IG_FUNC'] = data_eu['WESTBERRY_IG_FUNC']
    int_all['ZEU_WESTBERRY_NUT_TEMP_FUNC'] = data_eu['WESTBERRY_NUT_TEMP_FUNC']
    int_all['ZEU_CHLA'] = data_eu['CHLA']
    int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
    int_all['ZEU_TEMP'] = data_eu['TEMP']
    int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']

    data_mld = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
    int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
    int_all['MLD_WESTBERRY_IG_FUNC'] = data_mld['WESTBERRY_IG_FUNC']
    int_all['MLD_WESTBERRY_NUT_TEMP_FUNC'] = data_mld['WESTBERRY_NUT_TEMP_FUNC']
    int_all['MLD_CHLA'] = data_mld['CHLA']
    int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
    int_all['MLD_TEMP'] = data_mld['TEMP']
    int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']

    od = abs(1 / xds.KD_PAR)
    data_od = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
    int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
    int_all['OD1_WESTBERRY_IG_FUNC'] = data_od['WESTBERRY_IG_FUNC']
    int_all['OD1_WESTBERRY_NUT_TEMP_FUNC'] = data_od['WESTBERRY_NUT_TEMP_FUNC']
    int_all['OD1_CHLA'] = data_od['CHLA']
    int_all['OD1_CPHYTO'] = data_od['CPHYTO']
    int_all['OD1_TEMP'] = data_od['TEMP']
    int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
    
    int_all['DATETIME'] = xds.DATETIME
    int_all['LATITUDE'] = xds.LATITUDE
    int_all['LONGITUDE'] = xds.LONGITUDE
    int_all['ZEU'] = xds.ZEU
    int_all['MLD'] = xds.MLD
    int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
    int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG
    int_all['BIOME'] = xds.BIOME

    if int_all.N_PROF.shape[0] == 0:
        continue

    xds.to_netcdf(
        f"{outdir}/{name}.nc",
        format="NETCDF4",
        encoding=bgc.make_encoding(xds, complevel=1),
    )
    
    int_all.to_netcdf(
        f"{outdir_npp}/{name}_NPP.nc",
        format="NETCDF4",
        encoding=bgc.make_encoding(int_all, complevel=1),
    )

100%|█████████████████████████████████████████████████████████████████████████████████| 362/362 [01:32<00:00,  3.93it/s]


#### Apply Climatology - No Irradiance Sensors

In [12]:
fname = '../DATA/Output Final/Profiles_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITHOUT_IRRADIANCE/*.nc'
flist = sorted(glob.glob(fname))

outdir = Path('../DATA/Output Final/Profiles_SOCA/KD490/XING_CLIM/DAILY_WITHOUT_IRRADIANCE')
outdir_npp = Path('../DATA/Output Final/NPP_SOCA/KD490/XING_CLIM/DAILY_WITHOUT_IRRADIANCE')

outdir.mkdir(parents=True, exist_ok=True)
outdir_npp.mkdir(parents=True, exist_ok=True)

# Loop over each file in flist
for file in tqdm(flist):
    # Get the output file path
    out_file = os.path.join(outdir, os.path.basename(file))  # You can modify this if you want a different naming convention
    
    # Check if the file has already been processed
    if os.path.exists(out_file):
        #print(f"Skipping {file}, already processed.")
        continue  # Skip this file and move to the next one

    name = os.path.basename(file).split('.')[0]

    xds = xr.open_dataset(file)

    xds["CHLA"] = xds["CHLA"].transpose("N_PROF", "PRES")
    
    # --------------------------------------------------
    # 1. Extract biome and month per profile
    # --------------------------------------------------
    biome = np.nan_to_num(xds["BIOME"].values, nan=-1).astype(int)
    month = xds["DATETIME"].dt.month.values.astype(int)
    
    # --------------------------------------------------
    # 2. Vectorised lookup of monthly biome factor
    # --------------------------------------------------
    lookup = MONTHLY_CLIM.stack().rename("CHLA_SLOPE_FACTOR").reset_index()
    lookup = lookup.rename(columns={"level_1": "MONTH"})
    
    factor_df = pd.DataFrame({
        "N_PROF": np.arange(xds.sizes["N_PROF"]),
        "BIOME": biome,
        "MONTH": month,})
    
    factor_df = factor_df.merge(
        lookup,
        on=["BIOME", "MONTH"],
        how="left",)
    
    slope = factor_df["CHLA_SLOPE_FACTOR"].to_numpy(dtype="float32")

    # --------------------------------------------------
    # Skip file if no valid CHLA slope factor was found
    # --------------------------------------------------
    if np.all(~np.isfinite(slope)):
        #print(f"Skipping {Path(file).name}: no valid CHLA_SLOPE_FACTOR found")
        xds.close()
        continue
    
    # --------------------------------------------------
    # 3. Assign factor to dataset
    # --------------------------------------------------
    xds = xds.assign(CHLA_SLOPE_FACTOR=("N_PROF", slope))
    
    # --------------------------------------------------
    # 4. Remove Roesler factor, apply Xing biome/month factor
    # --------------------------------------------------
    xds["CHLA"] = xds["CHLA"] * 2.0
    xds["CHLA"] = xds["CHLA"] * xds["CHLA_SLOPE_FACTOR"]
    
    # --------------------------------------------------
    # 5. Clean corrected chlorophyll
    # --------------------------------------------------
    xds = bgc.clean_range_filter(xds, "CHLA", 0.014, 50)

    # Apply NPP algorithms
    ds = xds.sel(PRES=slice(0, 199))
    result = xr.apply_ufunc(bgc.cbpm_argo,
                            ds['CHLA'],
                            ds['CPHYTO'],
                            ds['PAR_DAILY'],
                            ds['DATETIME'].dt.dayofyear,
                            ds['LATITUDE'],
                            input_core_dims=[['PRES'], ['PRES'], [], [], []],
                            output_core_dims=[['PRES'], ['PRES'], ['PRES'], ['PRES']],
                            vectorize=True,
                            dask='parallelized',
                            output_dtypes=[float] * 4)

    xds = xds.assign(WESTBERRY_CBPM=result[0],
                     WESTBERRY_MU=result[1],
                     WESTBERRY_NUT_TEMP_FUNC=result[2],
                     WESTBERRY_IG_FUNC=result[3])

    xds['WESTBERRY_CBPM'] = xds['WESTBERRY_CBPM'].where(xds['WESTBERRY_CBPM'] >= 0, np.nan)

    var_list = ['CHLA', 'CPHYTO', 'WESTBERRY_CBPM']
        
    # Integrate to 200 m
    int_all = bgc.integrate_along_pres_multiple_vars(
        xds.where(xds.PRES <= 199, drop=False), var_list
    ).rename({v: f"{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to Zeu
    zeu_mask = xds.PRES <= xds.ZEU.broadcast_like(xds.PRES)
    int_zeu = bgc.integrate_along_pres_multiple_vars(
        xds.where(zeu_mask, drop=False), var_list
    ).rename({v: f"ZEU_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Integrate to MLD
    mld_mask = xds.PRES <= xds.MLD.broadcast_like(xds.PRES)
    int_mld = bgc.integrate_along_pres_multiple_vars(
        xds.where(mld_mask, drop=False), var_list
    ).rename({v: f"MLD_{v}_INT" for v in var_list}).where(lambda ds: ds != 0, np.nan)
    
    # Merge safely
    int_all = xr.merge([int_all, int_zeu, int_mld], join='outer')

    data_eu = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.ZEU, drop=False).mean(dim='PRES')
    int_all['ZEU_WESTBERRY_CBPM'] = data_eu['WESTBERRY_CBPM']
    int_all['ZEU_WESTBERRY_IG_FUNC'] = data_eu['WESTBERRY_IG_FUNC']
    int_all['ZEU_WESTBERRY_NUT_TEMP_FUNC'] = data_eu['WESTBERRY_NUT_TEMP_FUNC']
    int_all['ZEU_CHLA'] = data_eu['CHLA']
    int_all['ZEU_CPHYTO'] = data_eu['CPHYTO']
    int_all['ZEU_TEMP'] = data_eu['TEMP']
    int_all['ZEU_DOWNWELLING_PAR'] = data_eu['DOWNWELLING_PAR']

    data_mld = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= xds.MLD, drop=False).mean(dim='PRES')
    int_all['MLD_WESTBERRY_CBPM'] = data_mld['WESTBERRY_CBPM']
    int_all['MLD_WESTBERRY_IG_FUNC'] = data_mld['WESTBERRY_IG_FUNC']
    int_all['MLD_WESTBERRY_NUT_TEMP_FUNC'] = data_mld['WESTBERRY_NUT_TEMP_FUNC']
    int_all['MLD_CHLA'] = data_mld['CHLA']
    int_all['MLD_CPHYTO'] = data_mld['CPHYTO']
    int_all['MLD_TEMP'] = data_mld['TEMP']
    int_all['MLD_DOWNWELLING_PAR'] = data_mld['DOWNWELLING_PAR']

    od = abs(1 / xds.KD_PAR)
    data_od = xds[['WESTBERRY_CBPM', 
                   'WESTBERRY_IG_FUNC', 
                   'WESTBERRY_NUT_TEMP_FUNC', 
                   'CHLA', 
                   'CPHYTO', 
                   'TEMP', 
                   'DOWNWELLING_PAR']].where(xds.PRES <= od, drop=False).mean(dim='PRES')
    int_all['OD1_WESTBERRY_CBPM'] = data_od['WESTBERRY_CBPM']
    int_all['OD1_WESTBERRY_IG_FUNC'] = data_od['WESTBERRY_IG_FUNC']
    int_all['OD1_WESTBERRY_NUT_TEMP_FUNC'] = data_od['WESTBERRY_NUT_TEMP_FUNC']
    int_all['OD1_CHLA'] = data_od['CHLA']
    int_all['OD1_CPHYTO'] = data_od['CPHYTO']
    int_all['OD1_TEMP'] = data_od['TEMP']
    int_all['OD1_DOWNWELLING_PAR'] = data_od['DOWNWELLING_PAR']
    
    int_all['DATETIME'] = xds.DATETIME
    int_all['LATITUDE'] = xds.LATITUDE
    int_all['LONGITUDE'] = xds.LONGITUDE
    int_all['ZEU'] = xds.ZEU
    int_all['MLD'] = xds.MLD
    int_all['SOLAR_ELEVATION'] = xds.SOLAR_ELEVATION
    int_all['QUENCHING_FLAG'] = xds.QUENCHING_FLAG
    int_all['BIOME'] = xds.BIOME

    if int_all.N_PROF.shape[0] == 0:
        continue

    xds.to_netcdf(
        f"{outdir}/{name}.nc",
        format="NETCDF4",
        encoding=bgc.make_encoding(xds, complevel=1),
    )
    
    int_all.to_netcdf(
        f"{outdir_npp}/{name}_NPP.nc",
        format="NETCDF4",
        encoding=bgc.make_encoding(int_all, complevel=1),
    )

100%|█████████████████████████████████████████████████████████████████████████████████| 926/926 [02:38<00:00,  5.85it/s]


# Compiling Data Outputs

These are only used for comparisons to confirm that the SOCA-ML model worked as expected.

## SOCA PAR

In [16]:
par_dir = '../DATA/Output Final/Profiles_SOCA/ROESLER'

fname = f'{par_dir}/MEASURED/6901472_Sprof.nc'
ds1 = xr.open_dataset(fname)

fname = f'{par_dir}/MEASURED/6901493_Sprof.nc'
ds2 = xr.open_dataset(fname)

fname = f'{par_dir}/MEASURED/6901523_Sprof.nc'
ds3 = xr.open_dataset(fname)

fname = f'{par_dir}/MEASURED/6901773_Sprof.nc'
ds4 = xr.open_dataset(fname)

fname = f'{par_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901472_Sprof.nc'
ds1b = xr.open_dataset(fname)

fname = f'{par_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901493_Sprof.nc'
ds2b = xr.open_dataset(fname)

fname = f'{par_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901523_Sprof.nc'
ds3b = xr.open_dataset(fname)

fname = f'{par_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901773_Sprof.nc'
ds4b = xr.open_dataset(fname)

LC_Hour_Deci_1 = []

for i in range(ds1.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds1.LATITUDE[i], lng=ds1.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds1.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds1.DATETIME[i].values))
    LC_Hour_Deci_1.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_2 = []

for i in range(ds2.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds2.LATITUDE[i], lng=ds2.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds2.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds2.DATETIME[i].values))
    LC_Hour_Deci_2.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_3 = []

for i in range(ds3.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds3.LATITUDE[i], lng=ds3.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds3.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds3.DATETIME[i].values))
    LC_Hour_Deci_3.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_4 = []

for i in range(ds4.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds4.LATITUDE[i], lng=ds4.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds4.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds4.DATETIME[i].values))
    LC_Hour_Deci_4.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_1 = pd.Series(LC_Hour_Deci_1)
LC_Hour_Deci_1 = LC_Hour_Deci_1.repeat(250)

LC_Hour_Deci_2 = pd.Series(LC_Hour_Deci_2)
LC_Hour_Deci_2 = LC_Hour_Deci_2.repeat(250)

LC_Hour_Deci_3 = pd.Series(LC_Hour_Deci_3)
LC_Hour_Deci_3 = LC_Hour_Deci_3.repeat(250)

LC_Hour_Deci_4 = pd.Series(LC_Hour_Deci_4)
LC_Hour_Deci_4 = LC_Hour_Deci_4.repeat(250)

lval = 0.001
data1 = pd.DataFrame({
    'measured': ds1.DOWNWELLING_PAR[:, :250].to_series(),
    'modelled': ds1b.DOWNWELLING_PAR[:, :250].to_series(), 
    'hour': LC_Hour_Deci_1.values,
    'WMO':6901472}).dropna()
m = (data1.hour >= 8) & (data1.hour <= 18)
data1 = data1.loc[m]
m = (data1.modelled >= lval) & (data1.measured >= lval)
soca_globcolour_par_data1 = data1.loc[m]

data2 = pd.DataFrame({
    'measured': ds2.DOWNWELLING_PAR[:, :250].to_series(),
    'modelled': ds2b.DOWNWELLING_PAR[:, :250].to_series(), 
    'hour': LC_Hour_Deci_2.values, 
    'WMO': 6901493}).dropna()
m = (data2.hour >= 8) & (data2.hour <= 18)
data2 = data2.loc[m]
m = (data2.modelled >= lval) & (data2.measured >= lval)
soca_globcolour_par_data2 = data2.loc[m]

data3 = pd.DataFrame({
    'measured': ds3.DOWNWELLING_PAR[:, :250].to_series(),
    'modelled': ds3b.DOWNWELLING_PAR[:, :250].to_series(), 
    'hour': LC_Hour_Deci_3.values, 
    'WMO':6901523}).dropna()
m = (data3.hour >= 8) & (data3.hour <= 18)
data3 = data3.loc[m]
m = (data3.modelled >= lval) & (data3.measured >= lval)
soca_globcolour_par_data3 = data3.loc[m]

data4 = pd.DataFrame({
    'measured': ds4.DOWNWELLING_PAR[:, :250].to_series(),
    'modelled': ds4b.DOWNWELLING_PAR[:, :250].to_series(), 
    'hour': LC_Hour_Deci_4.values, 
    'WMO': 6901773}).dropna()
m = (data4.hour >= 8) & (data4.hour <= 18)
data4 = data4.loc[m]
m = (data4.modelled >= lval) & (data4.measured >= lval)
soca_globcolour_par_data4 = data4.loc[m]

soca_globcolour_par_data = pd.concat([soca_globcolour_par_data1, soca_globcolour_par_data2, soca_globcolour_par_data3, soca_globcolour_par_data4])

soca_globcolour_par_data.to_csv('../DATA/SOCA VALIDATION/soca_globcolour_par_data.csv')

## SOCA Ed490

In [18]:
ed_dir = '../DATA/Output Final/Profiles_SOCA/KD490/'

fname = f'{ed_dir}/MEASURED/6901472_Sprof.nc'
ds1 = xr.open_dataset(fname)

fname = f'{ed_dir}/MEASURED/6901493_Sprof.nc'
ds2 = xr.open_dataset(fname)

fname = f'{ed_dir}/MEASURED/6901523_Sprof.nc'
ds3 = xr.open_dataset(fname)

fname = f'{ed_dir}/MEASURED/6901773_Sprof.nc'
ds4 = xr.open_dataset(fname)

ed_dir = '../DATA/Output Final/Profiles_SOCA/ROESLER/'

fname = f'{ed_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901472_Sprof.nc'
ds1b = xr.open_dataset(fname)

fname = f'{ed_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901493_Sprof.nc'
ds2b = xr.open_dataset(fname)

fname = f'{ed_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901523_Sprof.nc'
ds3b = xr.open_dataset(fname)

fname = f'{ed_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901773_Sprof.nc'
ds4b = xr.open_dataset(fname)

LC_Hour_Deci_1 = []

for i in range(ds1.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds1.LATITUDE[i], lng=ds1.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds1.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds1.DATETIME[i].values))
    LC_Hour_Deci_1.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_2 = []

for i in range(ds2.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds2.LATITUDE[i], lng=ds2.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds2.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds2.DATETIME[i].values))
    LC_Hour_Deci_2.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_3 = []

for i in range(ds3.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds3.LATITUDE[i], lng=ds3.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds3.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds3.DATETIME[i].values))
    LC_Hour_Deci_3.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_4 = []

for i in range(ds4.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds4.LATITUDE[i], lng=ds4.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds4.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds4.DATETIME[i].values))
    LC_Hour_Deci_4.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_1 = pd.Series(LC_Hour_Deci_1)
LC_Hour_Deci_1 = LC_Hour_Deci_1.repeat(250)

LC_Hour_Deci_2 = pd.Series(LC_Hour_Deci_2)
LC_Hour_Deci_2 = LC_Hour_Deci_2.repeat(250)

LC_Hour_Deci_3 = pd.Series(LC_Hour_Deci_3)
LC_Hour_Deci_3 = LC_Hour_Deci_3.repeat(250)

LC_Hour_Deci_4 = pd.Series(LC_Hour_Deci_4)
LC_Hour_Deci_4 = LC_Hour_Deci_4.repeat(250)

lval = 0.00001
data1 = pd.DataFrame({
    'measured': ds1.DOWN_IRRADIANCE490[:, :250].to_series(),
    'modelled': ds1b.DOWN_IRRADIANCE490[:, :250].to_series(), 
    'hour': LC_Hour_Deci_1.values,
    'WMO':6901472}).dropna()
m = (data1.hour >= 8) & (data1.hour <= 18)
data1 = data1.loc[m]
m = (data1.modelled >= lval) & (data1.measured >= lval)
soca_globcolour_ed490_data1 = data1.loc[m]

data2 = pd.DataFrame({
    'measured': ds2.DOWN_IRRADIANCE490[:, :250].to_series(),
    'modelled': ds2b.DOWN_IRRADIANCE490[:, :250].to_series(), 
    'hour': LC_Hour_Deci_2.values, 
    'WMO': 6901493}).dropna()
m = (data2.hour >= 8) & (data2.hour <= 18)
data2 = data2.loc[m]
m = (data2.modelled >= lval) & (data2.measured >= lval)
soca_globcolour_ed490_data2 = data2.loc[m]

data3 = pd.DataFrame({
    'measured': ds3.DOWN_IRRADIANCE490[:, :250].to_series(),
    'modelled': ds3b.DOWN_IRRADIANCE490[:, :250].to_series(), 
    'hour': LC_Hour_Deci_3.values, 
    'WMO':6901523}).dropna()
m = (data3.hour >= 8) & (data3.hour <= 18)
data3 = data3.loc[m]
m = (data3.modelled >= lval) & (data3.measured >= lval)
soca_globcolour_ed490_data3 = data3.loc[m]

data4 = pd.DataFrame({
    'measured': ds4.DOWN_IRRADIANCE490[:, :250].to_series(),
    'modelled': ds4b.DOWN_IRRADIANCE490[:, :250].to_series(), 
    'hour': LC_Hour_Deci_4.values, 
    'WMO': 6901773}).dropna()
m = (data4.hour >= 8) & (data4.hour <= 18)
data4 = data4.loc[m]
m = (data4.modelled >= lval) & (data4.measured >= lval)
soca_globcolour_ed490_data4 = data4.loc[m]

soca_globcolour_ed490_data = pd.concat([soca_globcolour_ed490_data1, soca_globcolour_ed490_data2, soca_globcolour_ed490_data3, soca_globcolour_ed490_data4])

soca_globcolour_ed490_data.to_csv('../DATA/SOCA VALIDATION/soca_globcolour_ed490_data.csv')

## SOCA NPP

In [19]:
npp_dir = '../DATA/Output Final/Profiles_SOCA/ROESLER/'

fname = f'{npp_dir}/MEASURED/6901472_Sprof.nc'
ds1 = xr.open_dataset(fname)
ds1 = ds1.where(ds1.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/MEASURED/6901493_Sprof.nc'
ds2 = xr.open_dataset(fname)
ds2 = ds2.where(ds2.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/MEASURED/6901523_Sprof.nc'
ds3 = xr.open_dataset(fname)
ds3 = ds3.where(ds3.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/MEASURED/6901773_Sprof.nc'
ds4 = xr.open_dataset(fname)
ds4 = ds4.where(ds4.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901472_Sprof.nc'
ds1b = xr.open_dataset(fname)
ds1b = ds1b.where(ds1b.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901493_Sprof.nc'
ds2b = xr.open_dataset(fname)
ds2b = ds2b.where(ds2b.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901523_Sprof.nc'
ds3b = xr.open_dataset(fname)
ds3b = ds3b.where(ds3b.SOLAR_ELEVATION > 0, drop=True)

fname = f'{npp_dir}/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/6901773_Sprof.nc'
ds4b = xr.open_dataset(fname)
ds4b = ds4b.where(ds4b.SOLAR_ELEVATION > 0, drop=True)

LC_Hour_Deci_1 = []

for i in range(ds1.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds1.LATITUDE[i], lng=ds1.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds1.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds1.DATETIME[i].values))
    LC_Hour_Deci_1.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_2 = []

for i in range(ds2.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds2.LATITUDE[i], lng=ds2.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds2.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds2.DATETIME[i].values))
    LC_Hour_Deci_2.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_3 = []

for i in range(ds3.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds3.LATITUDE[i], lng=ds3.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds3.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds3.DATETIME[i].values))
    LC_Hour_Deci_3.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_4 = []

for i in range(ds4.LATITUDE.shape[0]):
    timezone_str = tf.certain_timezone_at(lat=ds4.LATITUDE[i], lng=ds4.LONGITUDE[i])
    timezone = pytz.timezone(timezone_str)
    LC = pd.to_datetime(ds4.DATETIME[i].values) + timezone.utcoffset(pd.to_datetime(ds4.DATETIME[i].values))
    LC_Hour_Deci_4.append(LC.hour + (LC.minute / 60))

LC_Hour_Deci_1 = pd.Series(LC_Hour_Deci_1)

LC_Hour_Deci_2 = pd.Series(LC_Hour_Deci_2)

LC_Hour_Deci_3 = pd.Series(LC_Hour_Deci_3)

LC_Hour_Deci_4 = pd.Series(LC_Hour_Deci_4)

LC_Hour_Deci_1 = LC_Hour_Deci_1.repeat(200)
LC_Hour_Deci_2 = LC_Hour_Deci_2.repeat(200)
LC_Hour_Deci_3 = LC_Hour_Deci_3.repeat(200)
LC_Hour_Deci_4 = LC_Hour_Deci_4.repeat(200)

# Upper and lower limits for comparison
uval = 500
lval = 0.00001

data1 = pd.DataFrame({
    'measured': ds1.WESTBERRY_CBPM[:, :200].to_series().values,
    'modelled': ds1b.WESTBERRY_CBPM[:, :200].to_series().values, 
    'hour': LC_Hour_Deci_1.values,
    'WMO':6901472}).dropna()
m = (data1.hour >= 8) & (data1.hour <= 18)
data1 = data1.loc[m]
m = (data1.modelled <= uval) & (data1.measured <= uval)
data1 = data1.loc[m]
m = (data1.modelled >= lval) & (data1.measured >= lval)
soca_globcolour_westberry_cbpm_data1 = data1.loc[m]

data2 = pd.DataFrame({
    'measured': ds2.WESTBERRY_CBPM[:, :200].to_series().values,
    'modelled': ds2b.WESTBERRY_CBPM[:, :200].to_series().values, 
    'hour': LC_Hour_Deci_2.values, 
    'WMO': 6901493}).dropna()
m = (data2.hour >= 8) & (data2.hour <= 18)
data2 = data2.loc[m]
m = (data2.modelled <= uval) & (data2.measured <= uval)
data2 = data2.loc[m]
m = (data2.modelled >= lval) & (data2.measured >= lval)
soca_globcolour_westberry_cbpm_data2 = data2.loc[m]

data3 = pd.DataFrame({
    'measured': ds3.WESTBERRY_CBPM[:, :200].to_series().values,
    'modelled': ds3b.WESTBERRY_CBPM[:, :200].to_series().values, 
    'hour': LC_Hour_Deci_3.values, 
    'WMO':6901523}).dropna()
m = (data3.hour >= 8) & (data3.hour <= 18)
data3 = data3.loc[m]
m = (data3.modelled <= uval) & (data3.measured <= uval)
data3 = data3.loc[m]
m = (data3.modelled >= lval) & (data3.measured >= lval)
soca_globcolour_westberry_cbpm_data3 = data3.loc[m]

data4 = pd.DataFrame({
    'measured': ds4.WESTBERRY_CBPM[:, :200].to_series().values,
    'modelled': ds4b.WESTBERRY_CBPM[:, :200].to_series().values, 
    'hour': LC_Hour_Deci_4.values, 
    'WMO': 6901773}).dropna()
m = (data4.hour >= 8) & (data4.hour <= 18)
data4 = data4.loc[m]
m = (data4.modelled <= uval) & (data4.measured <= uval)
data4 = data4.loc[m]
m = (data4.modelled >= lval) & (data4.measured >= lval)
soca_globcolour_westberry_cbpm_data4 = data4.loc[m]

soca_globcolour_westberry_cbpm_data = pd.concat([soca_globcolour_westberry_cbpm_data1, soca_globcolour_westberry_cbpm_data2, 
                                                 soca_globcolour_westberry_cbpm_data3, soca_globcolour_westberry_cbpm_data4])

soca_globcolour_westberry_cbpm_data.to_csv('../DATA/SOCA VALIDATION/soca_globcolour_westberry_cbpm_data.csv')

## NPP Data

Here we export all the integrated data for further analysis.

### Measured PAR

#### Roesler

In [20]:
fname = '../DATA/Output Final/NPP_SOCA/ROESLER/MEASURED/*.nc'
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_FLOATS_MEASURED_PAR.nc")


100%|████████████████████████████████████████████████████████████████████████████████| 362/362 [00:02<00:00, 127.88it/s]


#### Kd490

In [21]:
fname = '../DATA/Output Final/NPP_SOCA/KD490/MEASURED/*.nc'
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_FLOATS_KD490_MEASURED_PAR.nc")


100%|████████████████████████████████████████████████████████████████████████████████| 339/339 [00:02<00:00, 117.85it/s]


### Modelled PAR

#### Roesler

In [22]:
fname = "../DATA/Output Final/NPP_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITH_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_GLOBCOLOUR_ROESLER_FLOATS_WITH_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 362/362 [00:03<00:00, 113.99it/s]


In [23]:
fname = "../DATA/Output Final/NPP_SOCA/ROESLER/GLOBCOLOUR/DAILY_WITHOUT_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_GLOBCOLOUR_ROESLER_FLOATS_NO_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 926/926 [00:08<00:00, 110.16it/s]


#### Kd490

##### Float Specific

In [24]:
fname = "../DATA/Output Final/NPP_SOCA/KD490/XING/DAILY_WITH_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_XING_KD490_FLOATS_WITH_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 333/333 [00:02<00:00, 123.64it/s]


In [25]:
fname = "../DATA/Output Final/NPP_SOCA/KD490/XING/DAILY_WITHOUT_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_XING_KD490_FLOATS_NO_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 772/772 [00:06<00:00, 117.73it/s]


##### Climatology

In [26]:
fname = "../DATA/Output Final/NPP_SOCA/KD490/XING_CLIM/DAILY_WITH_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_XING_CLIM_KD490_FLOATS_WITH_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 274/274 [00:02<00:00, 116.96it/s]


In [27]:
fname = "../DATA/Output Final/NPP_SOCA/KD490/XING_CLIM/DAILY_WITHOUT_IRRADIANCE/*.nc"
flist = sorted(glob.glob(fname))

ds1 = []
fl = []

for file in tqdm(flist):
    fl.append(file.split('/')[-1].split('_')[0])
    ds1.append(xr.open_dataset(file))

ds1 = xr.concat(ds1, dim=xr.DataArray(fl, dims='FLOAT', name='FLOAT'), join='outer')

ds1.to_netcdf("../DATA/BGC_ARGO_NPP_RESULTS_XING_CLIM_KD490_FLOATS_NO_PAR.nc")

100%|████████████████████████████████████████████████████████████████████████████████| 899/899 [00:07<00:00, 115.38it/s]
